In [3]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
#  Smart APS V9  —  5-Day Target Inventory System
# =============================================================

PLANNING_DATE = date(2026, 4, 6)
INDENT_MONTH  = date(2026, 4,  1)

AVAILABLE_HOURS      = 22
MIN_RUN_HOURS        = 4
MACHINE_STATE_FILE   = "machine_state.json"

MIN_DAILY_INDENT     = 150
MIN_INDENT_HOURS     = 4.0

SAFETY_DAYS  = 3
TARGET_DAYS  = 5

# ─────────────────────────────────────────────────────────────
# CHANGE 1: OPD caps now strictly follow the S0/S1/S2/S3
#           scenario framework shown in the slide:
#   S0 — CRITICAL LOW  (<1 day)  → cap 1.5× daily max
#   S1 — MIXED RISK    (1-3 days)→ cap 3× daily max
#   S2 — BELOW TARGET  (4 days)  → cap 4× daily max
#   S3 — HEALTHY STOCK (5+ days) → cap 5× daily max
# ─────────────────────────────────────────────────────────────
OPD_SCENARIO_0 = 1.5
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

# ─────────────────────────────────────────────────────────────
# CHANGE 4: 90% utilisation is the SOFT CEILING (breakdown
#           buffer).  Machines should not be loaded above 90%
#           in normal scheduling.  The enforcer still tries to
#           reach 90% on every machine but will NOT push beyond
#           it unless literally nothing else can be scheduled.
# ─────────────────────────────────────────────────────────────
UTIL_TARGET_PCT   = 90.0   # soft ceiling — breakdown buffer
UTIL_HARD_CAP_PCT = 100.0  # absolute hard cap (never exceeded)

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
terminal_path   = "C:/Users/Ex0164/Important codes/terminals and raw marterial - vt.xlsx"   # ← NEW: terminal file
output_path     = f"Smart_APS_V9_Plan_{PLANNING_DATE.strftime('%Y%m%d')} 1terminal logic.xlsx"

# ─────────────────────────────────────────────────────────────
# TERMINAL SHEET NAMES (edit if your sheet names differ)
#   Sheet 1: Part → required terminals  (one row per part,
#            Part column + one column per terminal, 1 = required)
#   Sheet 2: Currently unavailable terminals with quantities
# ─────────────────────────────────────────────────────────────
TERMINAL_PARTS_SHEET       = "Part_Terminals"      # Part → terminals matrix
TERMINAL_UNAVAILABLE_SHEET = "Unavailable_Terminals"  # Terminal ID + qty_unavailable

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(1 for d in range(1, total + 1) if date(year, month, d).weekday() == 6)
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V9  —  5-Day Target Inventory System")
print(f"  Planning date : {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"  Safety floor  : {SAFETY_DAYS} days  |  Target ceiling : {TARGET_DAYS} days")
print(f"  Util soft cap : {UTIL_TARGET_PCT}%  (breakdown buffer)")
print(f"  Terminal gate : ACTIVE — parts blocked if any required terminal is unavailable")
print(f"{'='*65}\n")

print("Loading data...")
vt_parts_raw         = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix            = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw            = pd.read_excel(changeover_path, sheet_name="VT_Changeover")
vt_machine_count_raw = pd.read_excel(matrix_path,     sheet_name="VT_Machine_Part_Count")

# ── Terminal data ──────────────────────────────────────────────
# Sheet 1 — Part_Terminals: rows = parts, columns = terminals
#   Format:  Part | T1 | T2 | T3 | T4 | ...
#   Cell = 1 (or any truthy value) means that terminal is required
#
# Sheet 2 — Unavailable_Terminals:
#   Format:  Terminal_ID | Qty_Unavailable | (optional) Reason
#   A terminal with Qty_Unavailable >= its total count is fully blocked.
#   If a terminal appears here with qty >= 1 it is considered UNAVAILABLE
#   (we treat each terminal as a single unit unless qty = 0 or row absent).
# ──────────────────────────────────────────────────────────────
try:
    _terminal_parts_raw    = pd.read_excel(terminal_path, sheet_name=TERMINAL_PARTS_SHEET)
    _terminal_unavail_raw  = pd.read_excel(terminal_path, sheet_name=TERMINAL_UNAVAILABLE_SHEET)
    _terminal_data_loaded  = True
    print(f"  Terminal data loaded  ✓  "
          f"({len(_terminal_parts_raw)} part rows, "
          f"{len(_terminal_unavail_raw)} unavailable terminal records)")
except FileNotFoundError:
    _terminal_data_loaded = False
    _terminal_parts_raw   = pd.DataFrame()
    _terminal_unavail_raw = pd.DataFrame()
    print(f"  WARNING: terminal_data.xlsx not found at '{terminal_path}'")
    print(f"           Terminal gate is DISABLED — all parts treated as terminal-clear.")
except Exception as _e:
    _terminal_data_loaded = False
    _terminal_parts_raw   = pd.DataFrame()
    _terminal_unavail_raw = pd.DataFrame()
    print(f"  WARNING: Could not load terminal data ({_e})")
    print(f"           Terminal gate is DISABLED — all parts treated as terminal-clear.")

def find_col(df, name, sheet):
    match = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(f"Column '{name}' not found in sheet '{sheet}'.\nAvailable: {list(df.columns)}")
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")
vt_col_color     = find_col(vt_parts_raw, "Color",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
print(f"  Parts with valid rate   : {len(data_valid)}")

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

tools_available = {}
part_color      = {}

for _, row in data.iterrows():
    p = str(row["Material"]).strip()

    v = row[vt_col_tools]
    tools_available[p] = max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1

    c = row[vt_col_color]
    part_color[p] = str(c).strip().upper() if pd.notna(c) and str(c).strip() not in ("", "nan") else "UNKNOWN"

color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)
print(f"  Distinct colours        : {len(color_groups)}")
for col, pts in sorted(color_groups.items()):
    print(f"    {col:<20} → {len(pts)} part(s)")

indent_daily = {p: round(qty / WORKING_DAYS, 4) for p, qty in indent_monthly.items()}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

COLOR_PURGE_HRS = 10 / 60.0   # 10 minutes expressed in hours


# =============================================================
# TERMINAL AVAILABILITY SYSTEM  (consumption-based model)
# =============================================================
#
# KEY DESIGN DECISIONS (confirmed by user):
#   • Terminals are RAW MATERIAL — 1 terminal consumed per piece.
#   • The terminal file's qty column = CURRENT STOCK (opening stock
#     for today).  This is NOT "qty unavailable" — it IS the stock.
#   • After planning, closing stock = opening stock − planned qty.
#   • A part is TERMINAL BLOCKED if stock of any required terminal = 0.
#   • Criticality thresholds (ALL categories — Runner, Repeater, Stranger):
#       HIGHLY CRITICAL : terminal stock covers < 75% of daily indent
#       CRITICAL        : terminal stock covers < 80% of daily indent
#       TERMINAL BLOCKED: terminal stock = 0  (hard block, cannot run)
#   • When a Runner is terminal-blocked AND inventory is critically low
#     (inv < SAFETY_DAYS * daily) → flag in a dedicated manual-intervention
#     alert sheet.
#
# FILE STRUCTURE (terminal_data.xlsx):
#   Sheet 1 — Part_Terminals:
#     Part | T1 | T2 | T3 | ...
#     Cell = 1 (or truthy) → that terminal is required for this part.
#     One terminal consumed per piece produced.
#
#   Sheet 2 — Unavailable_Terminals  (renamed concept: Terminal_Stock):
#     Terminal_ID | Current_Stock | (optional) Reason
#     Current_Stock = physical units available at start of day.
#     Stock = 0 → terminal is fully blocked.
# =============================================================

# Criticality thresholds
TERMINAL_CRITICAL_PCT       = 80.0   # stock covers < 80% daily indent → CRITICAL
TERMINAL_HIGHLY_CRITICAL_PCT = 75.0  # stock covers < 75% daily indent → HIGHLY CRITICAL


def _build_terminal_structures(parts_df, stock_df):
    """
    Parses Part_Terminals and Terminal_Stock sheets.

    Returns
    -------
    part_terminals  : {part: [terminal_id, ...]}
    terminal_stock  : {terminal_id: opening_stock (float)}
                      Current physical units available at start of day.
    unavailable_terminals : set of terminal IDs with stock == 0
    terminal_detail : {terminal_id: {"opening_stock", "reason"}}
    """
    part_terminals = {}

    if parts_df.empty:
        return part_terminals, {}, set(), {}

    cols     = list(parts_df.columns)
    part_col = cols[0]
    term_cols = cols[1:]

    for _, row in parts_df.iterrows():
        part = str(row[part_col]).strip()
        if not part or part.lower() in ("nan", ""):
            continue
        required = []
        for tc in term_cols:
            val = row[tc]
            if pd.notna(val) and str(val).strip() not in ("", "0", "0.0", "nan"):
                try:
                    if float(val) >= 1:
                        required.append(str(tc).strip())
                except (ValueError, TypeError):
                    if str(val).strip().lower() in ("yes", "y", "true", "✓", "x"):
                        required.append(str(tc).strip())
        part_terminals[part] = required

    # Parse stock sheet
    terminal_stock  = {}
    reason_map      = {}

    if not stock_df.empty:
        scols = [str(c).strip() for c in stock_df.columns]

        def _find(candidates):
            for n in candidates:
                for c in scols:
                    if c.strip().lower() == n.lower():
                        return c
            return None

        tid_col    = _find(["Terminal_ID", "Terminal", "terminal_id", "ID"])
        stock_col  = _find(["Current_Stock", "Stock", "Qty", "Quantity",
                             "Available_Qty", "Avail_Qty", "current_stock"])
        reason_col = _find(["Reason", "reason", "Note", "Remark"])

        if tid_col and stock_col:
            for _, row in stock_df.iterrows():
                tid = str(row[tid_col]).strip() if tid_col in stock_df.columns else ""
                if not tid or tid.lower() in ("nan", ""):
                    continue
                try:
                    stk = float(row[stock_col]) if pd.notna(row[stock_col]) else 0.0
                except (TypeError, ValueError):
                    stk = 0.0
                terminal_stock[tid] = stk
                if reason_col and reason_col in stock_df.columns:
                    rv = str(row[reason_col])
                    reason_map[tid] = "" if rv.lower() == "nan" else rv
        else:
            # Fallback: col1 = ID, col2 = stock
            for _, row in stock_df.iterrows():
                vals = [v for v in row.values if pd.notna(v)]
                if len(vals) >= 2:
                    tid = str(vals[0]).strip()
                    try:
                        stk = float(vals[1])
                    except (TypeError, ValueError):
                        stk = 0.0
                    if tid:
                        terminal_stock[tid] = stk

    # Build unavailable set (stock == 0)
    unavailable_set = {tid for tid, stk in terminal_stock.items() if stk <= 0}

    terminal_detail = {
        tid: {
            "opening_stock": terminal_stock.get(tid, 0.0),
            "reason":        reason_map.get(tid, ""),
        }
        for tid in terminal_stock
    }

    return part_terminals, terminal_stock, unavailable_set, terminal_detail


# Build global terminal structures
part_terminals, terminal_stock, unavailable_terminals, terminal_detail = \
    _build_terminal_structures(_terminal_parts_raw, _terminal_unavail_raw)

# ── Running stock tracker (mutable — updated as parts are planned) ──
# This tracks the CLOSING stock after each part is scheduled.
terminal_running_stock = dict(terminal_stock)   # starts = opening stock


def _consume_terminals(part, qty_produced):
    """
    Deduct terminal consumption for qty_produced pieces of `part`.
    Each required terminal loses qty_produced units (1 per piece).
    Updates terminal_running_stock in place.
    """
    for tid in part_terminals.get(part, []):
        if tid in terminal_running_stock:
            terminal_running_stock[tid] = max(
                0.0, terminal_running_stock[tid] - qty_produced
            )


def terminal_criticality(part, use_running_stock=False):
    """
    Returns (level: str, limiting_terminal: str, max_producible: float).

    level:
      "BLOCKED"         — stock = 0 for at least one required terminal
      "HIGHLY CRITICAL" — stock covers < TERMINAL_HIGHLY_CRITICAL_PCT% of daily indent
      "CRITICAL"        — stock covers < TERMINAL_CRITICAL_PCT% of daily indent
      "OK"              — stock sufficient for full daily indent

    max_producible: max pieces this part can make given current terminal stock.
    limiting_terminal: the terminal that is the tightest constraint.
    """
    required = part_terminals.get(part, [])
    if not required:
        return "OK", "—", float("inf")

    daily  = indent_daily.get(part, 0.0)
    stock  = terminal_running_stock if use_running_stock else terminal_stock

    min_stock      = float("inf")
    limiting_term  = "—"
    for tid in required:
        s = stock.get(tid, float("inf"))   # if not in stock dict → unlimited
        if s < min_stock:
            min_stock     = s
            limiting_term = tid

    if min_stock == float("inf"):
        return "OK", "—", float("inf")    # no stock data → treat as unlimited

    max_prod = min_stock   # 1 terminal per piece, so max pieces = min stock

    if max_prod <= 0:
        return "BLOCKED", limiting_term, 0.0

    if daily > 0:
        coverage_pct = (max_prod / daily) * 100
        if coverage_pct < TERMINAL_HIGHLY_CRITICAL_PCT:
            return "HIGHLY CRITICAL", limiting_term, max_prod
        if coverage_pct < TERMINAL_CRITICAL_PCT:
            return "CRITICAL", limiting_term, max_prod

    return "OK", limiting_term, max_prod


def is_terminal_blocked(part):
    """
    Returns (blocked: bool, reason: str).
    BLOCKED = stock of any required terminal is 0.
    """
    if not _terminal_data_loaded:
        return False, ""
    required = part_terminals.get(part, [])
    if not required:
        return False, ""
    blocked = [t for t in required if terminal_stock.get(t, float("inf")) <= 0]
    if blocked:
        details = [f"{t} (stock=0)" for t in blocked]
        return True, f"Terminal(s) with zero stock: {', '.join(details)}"
    return False, ""


def terminals_for_display(part):
    """All required terminals for a part, with opening stock."""
    req = part_terminals.get(part, [])
    if not req:
        return "—"
    return ", ".join(f"{t}(stk={terminal_stock.get(t,'?'):.0f})"
                     if t in terminal_stock else t
                     for t in req)


def blocked_terminals_for_display(part):
    """Only the zero-stock terminals for this part."""
    req     = part_terminals.get(part, [])
    blocked = [t for t in req if terminal_stock.get(t, float("inf")) <= 0]
    return ", ".join(blocked) or "—"


# ── Startup summary ─────────────────────────────────────────────
_parts_with_terminals  = sum(1 for t in part_terminals.values() if t)
_parts_no_terminals    = sum(1 for t in part_terminals.values() if not t)
_fully_blocked         = len(unavailable_terminals)

print(f"\n  Terminal system summary  (consumption model: 1 terminal per piece):")
print(f"    Parts mapped to terminals     : {_parts_with_terminals}")
print(f"    Parts with no terminal listed : {_parts_no_terminals}  (unrestricted)")
print(f"    Terminals with zero stock     : {_fully_blocked}")
print(f"    Criticality thresholds        : "
      f"HIGHLY CRITICAL <{TERMINAL_HIGHLY_CRITICAL_PCT}%  |  "
      f"CRITICAL <{TERMINAL_CRITICAL_PCT}%  of daily indent")
if unavailable_terminals:
    for tid in sorted(unavailable_terminals):
        affected = [p for p, ts in part_terminals.items() if tid in ts]
        print(f"      ✗  {tid:<22} stock=0  →  blocks {len(affected)} part(s)")
if terminal_stock:
    print(f"\n  Terminal opening stock (top constraints):")
    # show terminals where at least one part's daily indent is affected
    constraints = []
    for tid, stk in terminal_stock.items():
        parts_using = [p for p, ts in part_terminals.items() if tid in ts]
        max_daily   = max((indent_daily.get(p, 0) for p in parts_using), default=0)
        if max_daily > 0:
            cover_pct = (stk / max_daily * 100) if max_daily > 0 else 100
            constraints.append((cover_pct, tid, stk, max_daily))
    constraints.sort()
    for cover_pct, tid, stk, max_daily in constraints[:15]:
        flag = " ← HIGHLY CRITICAL" if cover_pct < TERMINAL_HIGHLY_CRITICAL_PCT else \
               " ← CRITICAL"        if cover_pct < TERMINAL_CRITICAL_PCT        else ""
        print(f"    {tid:<22} stock={stk:>8.0f}  "
              f"max_daily={max_daily:>8.0f}  covers={cover_pct:>6.1f}%{flag}")
else:
    print(f"    All terminals operational  ✓")
#   A Runner part with inventory >= SAFETY_DAYS (3 days) can
#   be deferred for ONE day.  It is flagged "RUNNER_DEFER_OK"
#   so the audit sheet explains why it was skipped today.
#   NOTE: This is a soft skip — if a machine has spare
#   capacity and nothing else can fill it, the enforcer may
#   still produce the runner (inventory build is fine).
# =============================================================
RUNNER_DEFER_INV_DAYS = SAFETY_DAYS  # skip runner if inv >= 3 days


def should_skip(part):
    """
    Returns (skip: bool, reason: str).

    CHANGE 1 — Thresholds clarified:
      Gate 4a: daily indent  <= MIN_DAILY_INDENT (150 pcs)  → SKIP
      Gate 4b: whole monthly indent runs in <= MIN_INDENT_HOURS (4 h) → SKIP

    CHANGE 5 — Runner deferral:
      Gate 5 (new): Runner part with inv >= SAFETY_DAYS * daily → DEFER today
        Rationale: keep buffer for uncertainty; must plan tomorrow.
        This is separate from "AT TARGET" (Gate 6).

    Gate 6: inv >= TARGET_DAYS * daily → AT CEILING, skip today.
    """
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    # Gate 4a
    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} threshold"

    # Gate 4b
    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Whole monthly indent = {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h threshold"

    inv      = inventory.get(part, 0.0)
    category = part_category.get(part, "Stranger") if "part_category" in globals() else "Stranger"

    # Gate 4c — Terminal availability (hard block: zero stock)
    # Only fully blocked (stock=0) parts are gated out here.
    # CRITICAL / HIGHLY CRITICAL parts still enter the scheduler
    # but are capped at their terminal max_producible qty.
    term_blocked, term_reason = is_terminal_blocked(part)
    if term_blocked:
        return True, f"TERMINAL BLOCKED — {term_reason}"

    # Gate 5 — Runner deferral (CHANGE 5)
    if category == "Runner" and daily > 0 and inv >= RUNNER_DEFER_INV_DAYS * daily:
        # Only defer if inventory is between SAFETY_DAYS and TARGET_DAYS
        # (if already at target Gate 6 will catch it)
        if inv < TARGET_DAYS * daily:
            return True, (f"Runner deferred — inv ({inv:.0f}) ≥ {RUNNER_DEFER_INV_DAYS}-day buffer "
                          f"({RUNNER_DEFER_INV_DAYS * daily:.0f} pcs). Plan tomorrow.")

    # Gate 6 — At 5-day ceiling
    if daily > 0 and inv >= TARGET_DAYS * daily:
        return True, (f"Inventory ({inv:.0f}) ≥ {TARGET_DAYS}-day target "
                      f"({TARGET_DAYS * daily:.0f} pcs) — at ceiling, skip today")

    return False, ""


def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

def build_machine_part_count(df):
    mpc = {}
    machine_col = next((c for c in df.columns if str(c).strip().lower() == "machine"), None)
    count_col   = next((c for c in df.columns if str(c).strip().lower() == "part_count"), None)
    if machine_col is None or count_col is None:
        print(f"  WARNING: VT_Machine_Part_Count sheet missing 'Machine' or 'Part_Count' column")
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(vt_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1
print(f"  Machine part counts loaded: {len(machine_part_count)} machines")

def build_category(df):
    cat     = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                print(f"  Machine state : file empty — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                print(f"  Machine state : file corrupt — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            print(f"  Machine state loaded  ({len(state)} machines with history)")
            return state
        except Exception as e:
            print(f"  Machine state : error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)


# =============================================================
# CHANGE 6: Per-part OPD scenario classification
#   Each part is assessed individually against the S0–S3
#   thresholds from the slide.  The scenario determines the
#   OPD cap for THAT PART only (not a global flag).
#
#   S0: inv < 1 day  → 1.5× cap  (Critical Low)
#   S1: 1 ≤ inv < 3  → 3.0× cap  (Mixed Risk)
#   S2: 3 ≤ inv < 5  → 4.0× cap  (Below Target)
#   S3: inv ≥ 5 days → 5.0× cap  (Healthy Stock)
#
#   The global scenario (S0/S1/S2/S3) is still computed for
#   the overall daily schedule summary.
# =============================================================
def part_opd_cap(part, current_inventory_dict=None):
    """
    Per-part OPD cap based on S0-S3 inventory health.
    Uses current_inventory_dict if provided (post-production),
    else falls back to opening inventory.
    """
    inv   = (current_inventory_dict or inventory).get(part, 0.0)
    daily = indent_daily.get(part, 0.0)
    days  = inv / daily if daily > 0 else 0.0

    if days < 1.0:
        return OPD_SCENARIO_0   # S0: Critical Low
    elif days < SAFETY_DAYS:
        return OPD_SCENARIO_1   # S1: Mixed Risk
    elif days < TARGET_DAYS:
        return OPD_SCENARIO_2   # S2: Below Target
    else:
        return OPD_SCENARIO_3   # S3: Healthy Stock (shouldn't normally reach here)


def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < SAFETY_DAYS)

    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {SAFETY_DAYS}-day safety floor"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (≥{SAFETY_DAYS} days safety floor)"


def opd_cap(scenario_id):
    """Global OPD cap (used for legacy calls). Prefer part_opd_cap() per part."""
    return {0: OPD_SCENARIO_0, 1: OPD_SCENARIO_1,
            2: OPD_SCENARIO_2, 3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_2)


def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_days = min(1.0, max(0.0, SAFETY_DAYS - days_cov) / SAFETY_DAYS)
        rows.append({"part": p, "inv": inv, "daily": daily,
                     "days_cov": days_cov, "cat": cat, "urgency_raw": gap_days})

    if not rows:
        return {}, []

    max_daily = max(r["daily"] for r in rows) or 1.0
    scores, score_rows = {}, []

    for r in rows:
        p              = r["part"]
        urgency_score  = r["urgency_raw"] * 100
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100
        final_score    = (W_URGENCY * urgency_score +
                          W_CATEGORY * category_score +
                          W_INDENT   * indent_score)
        scores[p] = round(final_score, 2)

        # Per-part scenario classification for the scores sheet
        inv_val  = r["inv"]
        daily_v  = r["daily"]
        days_v   = inv_val / daily_v if daily_v > 0 else 0
        if days_v < 1:
            inv_scenario = "S0-CRITICAL"
        elif days_v < SAFETY_DAYS:
            inv_scenario = "S1-MIXED_RISK"
        elif days_v < TARGET_DAYS:
            inv_scenario = "S2-BELOW_TARGET"
        else:
            inv_scenario = "S3-HEALTHY"

        score_rows.append({
            "Part":            p,
            "Category":        r["cat"],
            "Color":           part_color.get(p, "UNKNOWN"),
            "Tools":           tools_available.get(p, 1),
            "Required_Terminals": terminals_for_display(p),
            "Blocked_Terminals":  blocked_terminals_for_display(p),
            "Terminal_Status":    "BLOCKED" if is_terminal_blocked(p)[0] else "OK",
            "Inventory_Now":   round(r["inv"], 0),
            "Daily_Indent":    round(r["daily"], 2),
            "Days_Coverage":   round(r["days_cov"], 2),
            "Inv_Scenario":    inv_scenario,
            "OPD_Cap":         part_opd_cap(p),
            "Safety_Floor":    SAFETY_DAYS,
            "Target_Ceiling":  TARGET_DAYS,
            "Buffer_Status":   (
                "CRITICAL"     if r["days_cov"] < 1 else
                "BELOW_SAFETY" if r["days_cov"] < SAFETY_DAYS else
                "BUILDING"     if r["days_cov"] < TARGET_DAYS else
                "AT_TARGET"
            ),
            "Urgency_Score":   round(urgency_score, 1),
            "Category_Score":  category_score,
            "Indent_Score":    round(indent_score, 1),
            "Final_Score":     round(final_score, 2),
        })

    return scores, score_rows


# =============================================================
# DISPLACEMENT PASS
# =============================================================

def displace_for_zero_inv(part, machine_hours, machine_last_part, current_inventory,
                           plan, already_planned, priority_scores):
    daily    = indent_daily.get(part, 0)
    r_val    = rate.get(part, 1)
    category = part_category.get(part, "Stranger")
    score    = priority_scores.get(part, 0)
    compatible = vt_compat.get(part, [])

    if not compatible:
        return False

    # Only look at machines that are full (< MIN_RUN_HOURS free)
    candidate_machines = [
        m for m in compatible
        if round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4) < MIN_RUN_HOURS
    ]
    if not candidate_machines:
        return False

    best_machine     = None
    best_victim_row  = None
    best_victim_days = -1

    for m in candidate_machines:
        for row in [r for r in plan if r["Machine"] == m]:
            vpart  = row["Part"]
            vdaily = indent_daily.get(vpart, 0)
            vinv   = current_inventory.get(vpart, 0)
            vdays  = vinv / vdaily if vdaily > 0 else 999
            if vdays < SAFETY_DAYS or vinv <= 0:
                continue
            vrun = float(row.get("Run_Hours", 0))
            if vrun - MIN_RUN_HOURS < MIN_RUN_HOURS:
                continue
            if vdays > best_victim_days:
                best_victim_days = vdays
                best_victim_row  = row
                best_machine     = m

    if best_machine is None or best_victim_row is None:
        return False

    vpart    = best_victim_row["Part"]
    vr_val   = rate.get(vpart, 1)
    reduce_h = MIN_RUN_HOURS
    lost_qty = round(reduce_h * vr_val, 0)

    best_victim_row["Run_Hours"]      = round(float(best_victim_row["Run_Hours"]) - reduce_h, 3)
    best_victim_row["Production_Qty"] = round(float(best_victim_row["Production_Qty"]) - lost_qty, 0)
    best_victim_row["Total_Hrs_Used"] = round(
        float(best_victim_row.get("Changeover_Hrs", 0)) + float(best_victim_row["Run_Hours"]), 3)
    best_victim_row["Type"] = str(best_victim_row.get("Type", "Primary")) + " [DISPLACED]"

    current_inventory[vpart]    = round(current_inventory.get(vpart, 0) - lost_qty, 0)
    machine_hours[best_machine] = round(machine_hours.get(best_machine, 0) - reduce_h, 4)

    last = machine_last_part.get(best_machine)
    if last is None or last == part:
        co_hrs = 0.0
    else:
        base_co    = vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS)
        last_color = part_color.get(last, "UNKNOWN")
        new_color  = part_color.get(part, "UNKNOWN")
        purge      = (COLOR_PURGE_HRS
                      if last_color != new_color
                      and last_color != "UNKNOWN"
                      and new_color  != "UNKNOWN"
                      else 0.0)
        co_hrs = base_co + purge

    eff_free = round(AVAILABLE_HOURS - machine_hours.get(best_machine, 0) - co_hrs, 4)
    run_hrs  = max(MIN_RUN_HOURS, min(eff_free, MIN_RUN_HOURS))
    qty      = round(run_hrs * r_val, 0)

    machine_hours[best_machine]     = round(machine_hours.get(best_machine, 0) + co_hrs + run_hrs, 4)
    current_inventory[part]         = round(current_inventory.get(part, 0) + qty, 0)
    machine_last_part[best_machine] = part
    already_planned.add(part)

    plan.append({
        "Part":             part,
        "Color":            part_color.get(part, "UNKNOWN"),
        "Category":         category,
        "Machine":          best_machine,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(part, 0), 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Color_Purge":      "Yes" if co_hrs > vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS) else "No",
        "Type":             "Displacement [ZERO-INV PRIORITY]",
        "Role":             "Primary",
        "Tools_Available":  tools_available.get(part, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if qty >= daily else "NO — partial",
        "Stagger_Adjusted": "No",
        "Displaced_Victim": vpart,
        "Victim_Days_Stock":round(best_victim_days, 2),
        "Inv_Scenario":     _inv_scenario_label(part, inventory),
    })

    print(f"      ↳ DISPLACEMENT  {part:26s} → {best_machine:15s}  "
          f"freed from {vpart} ({best_victim_days:.1f} days stock)  "
          f"run={run_hrs:.2f}h  qty={qty:.0f}  "
          f"color={part_color.get(part,'?')}  purge={'Yes' if co_hrs > vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS) else 'No'}")
    return True


def _inv_scenario_label(part, inv_dict):
    inv   = inv_dict.get(part, 0.0)
    daily = indent_daily.get(part, 0.0)
    days  = inv / daily if daily > 0 else 0.0
    if days < 1:
        return "S0-CRITICAL"
    elif days < SAFETY_DAYS:
        return "S1-MIXED_RISK"
    elif days < TARGET_DAYS:
        return "S2-BELOW_TARGET"
    else:
        return "S3-HEALTHY"


# =============================================================
# MACHINE RANKER  — colour bonus added
# =============================================================

def rank_machines(part, machines_to_try, machine_hours, machine_last_part, inv_days):
    """
    CHANGE 4: available_hours capped at UTIL_TARGET_PCT (90%).
    A machine is considered "full" once it reaches 90% of 22h
    = 19.8h.  Scheduling beyond 19.8h is only allowed by the
    enforcer in Step 0 (floor extension, also capped at 90%).
    """
    category    = part_category.get(part, "Stranger")
    runner_lock = (category == "Runner" and inv_days <= 1.0)
    new_color   = part_color.get(part, "UNKNOWN")

    # Soft cap: 90% of available hours
    soft_cap_hrs = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)

    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        # CHANGE 4: use soft cap, not full 22h
        free = round(soft_cap_hrs - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue

        if last is None or last == part:
            co_hrs      = 0.0
            color_bonus = 0.0
        else:
            base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            last_color = part_color.get(last, "UNKNOWN")
            same_color = (last_color == new_color
                          and last_color != "UNKNOWN"
                          and new_color  != "UNKNOWN")
            purge      = 0.0 if same_color else (
                COLOR_PURGE_HRS
                if last_color != "UNKNOWN" and new_color != "UNKNOWN"
                else 0.0
            )
            co_hrs     = base_co + purge
            color_bonus = -0.08 if same_color else 0.0

        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue

        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0

        cost = count_score + co_penalty + util_penalty + same_part_bonus + color_bonus

        ranked.append((m, co_hrs, effective_free, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked, runner_lock


# =============================================================
# COLOUR-AWARE _co_hrs_for
# =============================================================

def _co_hrs_for(p, m, machine_last_part):
    last = machine_last_part.get(m)
    if last is None or last == p:
        return 0.0
    base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
    last_color = part_color.get(last, "UNKNOWN")
    new_color  = part_color.get(p,    "UNKNOWN")
    purge      = (COLOR_PURGE_HRS
                  if last_color != new_color
                  and last_color != "UNKNOWN"
                  and new_color  != "UNKNOWN"
                  else 0.0)
    return base_co + purge


# =============================================================
# TOOL-AWARE ASSIGNMENT  — CHANGES 2 & 3 applied here
#
# CHANGE 2: Repeater and Stranger parts → SINGLE TOOL ONLY.
#           No tool-expansion (Phase 2) for these categories.
#
# CHANGE 3: Repeater and Stranger parts → produce FULLY on ONE
#           machine.  If the shortfall cannot be met on a
#           single machine, produce as much as that machine
#           allows but do NOT spawn a second machine row.
#           (Partial production on one machine is acceptable;
#            splitting across two machines is NOT.)
# =============================================================

def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned, priority_scores):
    daily    = indent_daily.get(part, 0)
    monthly  = indent_monthly.get(part, 0)
    r_val    = rate.get(part, 1)
    inv_now  = current_inventory.get(part, 0)
    category = part_category.get(part, "Stranger")
    tools    = tools_available.get(part, 1)
    score    = priority_scores.get(part, 0)
    color    = part_color.get(part, "UNKNOWN")
    inv_days = inv_now / daily if daily > 0 else 999
    compatible = vt_compat.get(part, [])

    if not compatible:
        return []

    # ── Terminal production cap (CRITICAL / HIGHLY CRITICAL parts) ──
    # Blocked parts never reach here (gated in should_skip).
    # Parts with limited terminal stock are capped at max_producible.
    term_level, limiting_term, term_max_prod = terminal_criticality(part, use_running_stock=True)
    has_terminal_cap = (term_level in ("CRITICAL", "HIGHLY CRITICAL"))

    # CHANGE 2 & 3: Repeater and Stranger → single tool, single machine
    is_single_machine_only = (category in ("Repeater", "Stranger"))

    total_shortfall = max(0.0, daily - inv_now)
    hrs_for_full    = total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS
    hrs_for_full    = max(MIN_RUN_HOURS, hrs_for_full)

    # Cap shortfall at terminal max if applicable
    if has_terminal_cap and term_max_prod < total_shortfall:
        hrs_for_full = max(MIN_RUN_HOURS, term_max_prod / r_val if r_val > 0 else MIN_RUN_HOURS)
        total_shortfall = term_max_prod   # don't try to produce more than terminals allow

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0
    used_machines   = set()

    ranked, runner_lock = rank_machines(part, compatible, machine_hours, machine_last_part, inv_days)
    if not ranked:
        return []

    m1, co1, eff1, _ = ranked[0]
    run1 = min(eff1, hrs_for_full)
    run1 = max(run1, MIN_RUN_HOURS)
    qty1 = round(run1 * r_val, 0)

    machine_hours[m1]       = round(machine_hours.get(m1, 0) + co1 + run1, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
    machine_last_part[m1]   = part
    produced_so_far        += qty1
    tools_used             += 1
    used_machines.add(m1)
    already_planned.add(part)

    # Consume terminals for qty1 pieces
    _consume_terminals(part, qty1)

    indent_met_on_primary = (produced_so_far >= total_shortfall - 0.5)

    last_m1       = machine_state.get(m1)
    purge_applied = (last_m1 is not None and last_m1 != part
                     and part_color.get(last_m1, "UNKNOWN") != color
                     and part_color.get(last_m1, "UNKNOWN") != "UNKNOWN"
                     and color != "UNKNOWN")

    new_rows.append({
        "Part":             part,
        "Color":            color,
        "Category":         category,
        "Machine":          m1,
        "Run_Hours":        round(run1, 3),
        "Changeover_Hrs":   round(co1, 3),
        "Total_Hrs_Used":   round(co1 + run1, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty1,
        "Monthly_Indent":   round(monthly, 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co1 == 0 else "Yes",
        "Color_Purge":      "Yes" if purge_applied else "No",
        "Type":             "Primary" + (" [ZERO-INV]" if inv_now == 0 else ""),
        "Role":             "Primary",
        "Tools_Available":  tools,
        "Tools_Used":       1,
        "Runner_Lock":      "YES" if runner_lock else "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if indent_met_on_primary else "NO — shortfall remains",
        "Stagger_Adjusted": "No",
        "Inv_Scenario":     _inv_scenario_label(part, inventory),
        "Required_Terminals":   terminals_for_display(part),
        "Terminal_Criticality": term_level,
        "Limiting_Terminal":    limiting_term,
        "Terminal_Max_Prod":    round(term_max_prod, 0) if term_max_prod != float("inf") else "Unlimited",
    })

    # ─── PHASE 2: Tool expansion ───────────────────────────────
    # CHANGE 2 & 3: Repeater/Stranger → SKIP tool expansion entirely
    if not indent_met_on_primary and not is_single_machine_only:
        is_critical   = (inv_now == 0)
        tool_hard_cap = tools if is_critical else min(2, tools)

        while produced_so_far < (total_shortfall - 0.5) and tools_used < tool_hard_cap:
            shortfall_now = total_shortfall - produced_so_far
            hrs_needed    = max(MIN_RUN_HOURS, shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS)

            remaining_machines = [m for m in compatible if m not in used_machines]
            ranked_next, _ = rank_machines(part, remaining_machines, machine_hours,
                                           machine_last_part, inv_days)
            if not ranked_next:
                break

            mx, cox, effx, _ = ranked_next[0]
            run_x = min(effx, hrs_needed)
            run_x = max(run_x, MIN_RUN_HOURS)
            qty_x = round(run_x * r_val, 0)

            machine_hours[mx]       = round(machine_hours.get(mx, 0) + cox + run_x, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + qty_x, 0)
            machine_last_part[mx]   = part
            produced_so_far        += qty_x
            tools_used             += 1
            used_machines.add(mx)

            # Consume terminals for this expansion batch
            _consume_terminals(part, qty_x)

            indent_met_here = (produced_so_far >= total_shortfall - 0.5)

            last_mx       = machine_state.get(mx)
            purge_x       = (last_mx is not None and last_mx != part
                             and part_color.get(last_mx, "UNKNOWN") != color
                             and part_color.get(last_mx, "UNKNOWN") != "UNKNOWN"
                             and color != "UNKNOWN")

            new_rows.append({
                "Part":             part,
                "Color":            color,
                "Category":         category,
                "Machine":          mx,
                "Run_Hours":        round(run_x, 3),
                "Changeover_Hrs":   round(cox, 3),
                "Total_Hrs_Used":   round(cox + run_x, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty_x,
                "Monthly_Indent":   round(monthly, 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(part, 0), 0),
                "Changeover":       "No" if cox == 0 else "Yes",
                "Color_Purge":      "Yes" if purge_x else "No",
                "Type":             "Tool-Expansion",
                "Role":             f"Tool-Expansion (tool {tools_used}/{tools})",
                "Tools_Available":  tools,
                "Tools_Used":       tools_used,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            2,
                "Indent_Met":       "YES" if indent_met_here else "NO — shortfall remains",
                "Stagger_Adjusted": "No",
                "Inv_Scenario":     _inv_scenario_label(part, inventory),
                "Required_Terminals":   terminals_for_display(part),
                "Terminal_Criticality": term_level,
                "Limiting_Terminal":    limiting_term,
                "Terminal_Max_Prod":    round(term_max_prod, 0) if term_max_prod != float("inf") else "Unlimited",
            })

            crisis_tag = " [CRISIS — 3rd+ tool]" if tools_used >= 3 else ""
            print(f"      ↳ TOOL-EXP {part:26s} tool {tools_used}/{tool_hard_cap} → "
                  f"{mx:15s}  {run_x:.2f}h  qty={qty_x:.0f}  "
                  f"color={color}  {'COVERED ✓' if indent_met_here else 'still short'}{crisis_tag}")

    elif not indent_met_on_primary and is_single_machine_only:
        # CHANGE 3: Log that Repeater/Stranger is intentionally partial (no second machine)
        print(f"      ↳ SINGLE-MACHINE [{category}] {part:22s} → partial production "
              f"accepted  qty={qty1:.0f}  shortfall={(total_shortfall-qty1):.0f}  "
              f"(no tool expansion for {category})")

    # ─── PHASE 3: Inventory build ──────────────────────────────
    # CHANGE 6: Use per-part OPD cap based on inventory scenario
    cap_days    = part_opd_cap(part, current_inventory)
    inv_after   = current_inventory.get(part, 0)
    cap_qty     = cap_days * daily
    headroom_qty = max(0.0, cap_qty - inv_after)

    if headroom_qty > 0:
        # CHANGE 3: For Repeater/Stranger, only extend on the primary machine (new_rows[0])
        rows_to_extend = new_rows[:1] if is_single_machine_only else new_rows
        for row in rows_to_extend:
            if headroom_qty <= 0:
                break
            m      = row["Machine"]
            # CHANGE 4: respect 90% soft cap during inv build too
            soft_cap_hrs = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)
            free_m = round(soft_cap_hrs - machine_hours.get(m, 0), 4)
            if free_m < 0.05:
                continue
            extend_hrs = min(free_m, headroom_qty / r_val if r_val > 0 else 0)
            if extend_hrs < 0.05:
                continue
            extra_qty = round(extend_hrs * r_val, 0)

            row["Run_Hours"]      = round(float(row["Run_Hours"]) + extend_hrs, 3)
            row["Total_Hrs_Used"] = round(float(row["Changeover_Hrs"]) + float(row["Run_Hours"]), 3)
            row["Production_Qty"] = round(float(row["Production_Qty"]) + extra_qty, 0)
            row["Type"]           = str(row["Type"]) + "+InvBuild"

            machine_hours[m]        = round(machine_hours.get(m, 0) + extend_hrs, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
            headroom_qty           -= extra_qty

            print(f"      ↳ INV-BUILD {part:25s} on {m:15s}  "
                  f"+{extend_hrs:.2f}h  qty+={extra_qty:.0f}  "
                  f"(S{_scenario_id_for_days(inv_after/daily if daily>0 else 0)} cap: {cap_days:.1f} days)")

    for row in new_rows:
        row["Tools_Used"] = tools_used

    return new_rows


def _scenario_id_for_days(days):
    if days < 1:
        return 0
    elif days < SAFETY_DAYS:
        return 1
    elif days < TARGET_DAYS:
        return 2
    else:
        return 3


# =============================================================
# TOOL-CHANGER HELPERS
# =============================================================

def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i-1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i-1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)

def _extend_row_before(ev, wait_hrs, plan):
    # CHANGE 4: only extend up to 90% cap
    soft_cap_hrs = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)
    used_m = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == ev["machine"]
    )
    spare     = max(0.0, soft_cap_hrs - used_m)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CO: extended +{round(extend_by*60,1)}min to fill TC wait"
    return extend_by, extra


# =============================================================
# QUANTITY-BASED CO STAGGER
# =============================================================

MIN_CO_GAP_HRS = 20 / 60.0

def _finish_time_of_co(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    total = 0.0
    for row in plan:
        if row["Machine"] != m:
            continue
        if row is target:
            break
        total += float(row.get("Run_Hours") or 0)
        total += float(row.get("Changeover_Hrs") or 0)
    return round(total, 4)

def _last_row_before_co(ev, plan):
    return ev["row_before"]

def stagger_co_by_quantity(plan, machines, scenario_id, current_inventory):
    events = _collect_co_events(plan, machines)
    if len(events) < 2:
        return 0

    adjustments = 0
    max_passes  = len(events) * 2

    for _ in range(max_passes):
        for ev in events:
            ev["_ft"] = _finish_time_of_co(ev, plan)
        events.sort(key=lambda e: e["_ft"])

        conflict = None
        for i in range(len(events) - 1):
            co_dur_e = events[i]["co_duration"]
            required = co_dur_e + MIN_CO_GAP_HRS
            gap      = events[i+1]["_ft"] - events[i]["_ft"]
            if gap < required - 0.001:
                conflict = (events[i], events[i+1], gap, required)
                break

        if conflict is None:
            break

        ev_early, ev_late, gap, required_gap = conflict
        shortfall_hrs = required_gap - gap

        row_late  = _last_row_before_co(ev_late, plan)
        p_late    = row_late["Part"]
        r_late    = rate.get(p_late, 1)
        m_late    = ev_late["machine"]
        daily_l   = indent_daily.get(p_late, 0)
        inv_l     = current_inventory.get(p_late, 0)
        # CHANGE 6: use per-part OPD cap
        cap_qty   = part_opd_cap(p_late, current_inventory) * daily_l
        produced_l = float(row_late.get("Production_Qty") or 0)
        headroom  = max(0.0, cap_qty - inv_l)
        push_hrs  = shortfall_hrs
        push_qty  = round(push_hrs * r_late, 0)
        # CHANGE 4: check against soft cap
        soft_cap_hrs = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)
        used_m    = sum(float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
                        for r in plan if r["Machine"] == m_late)
        free_m    = max(0.0, soft_cap_hrs - used_m)

        can_push = (push_qty <= headroom and push_hrs <= free_m + 0.001 and r_late > 0)

        if can_push:
            row_late["Run_Hours"]      = round(float(row_late.get("Run_Hours") or 0) + push_hrs, 3)
            row_late["Production_Qty"] = round(produced_l + push_qty, 0)
            row_late["Total_Hrs_Used"] = round(
                float(row_late.get("Changeover_Hrs") or 0) + float(row_late["Run_Hours"]), 3)
            row_late["Stagger_Adjusted"] = (
                f"CO-stagger PUSH +{round(push_hrs*60,1)}min "
                f"gap={round(gap*60,1)}min need={round(required_gap*60,0):.0f}min")
            current_inventory[p_late] = round(current_inventory.get(p_late, 0) + push_qty, 0)
            adjustments += 1
            continue

        row_early  = _last_row_before_co(ev_early, plan)
        p_early    = row_early["Part"]
        r_early    = rate.get(p_early, 1)
        daily_e    = indent_daily.get(p_early, 0)
        inv_e      = current_inventory.get(p_early, 0)
        produced_e = float(row_early.get("Production_Qty") or 0)
        min_qty_e  = max(0.0, daily_e - inv_e)
        max_pull   = max(0.0, produced_e - min_qty_e)
        pull_hrs   = shortfall_hrs
        pull_qty   = round(pull_hrs * r_early, 0)

        can_pull = (pull_qty <= max_pull and r_early > 0
                    and produced_e - pull_qty >= MIN_RUN_HOURS * r_early)

        if can_pull:
            row_early["Run_Hours"]      = round(float(row_early.get("Run_Hours") or 0) - pull_hrs, 3)
            row_early["Production_Qty"] = round(produced_e - pull_qty, 0)
            row_early["Total_Hrs_Used"] = round(
                float(row_early.get("Changeover_Hrs") or 0) + float(row_early["Run_Hours"]), 3)
            row_early["Stagger_Adjusted"] = (
                f"CO-stagger PULL -{round(pull_hrs*60,1)}min "
                f"gap={round(gap*60,1)}min need={round(required_gap*60,0):.0f}min")
            current_inventory[p_early] = round(current_inventory.get(p_early, 0) - pull_qty, 0)
            adjustments += 1
            continue

        print(f"    [CO-STAGGER SKIP]  {ev_early['machine']} / {ev_late['machine']}  "
              f"gap={round(gap*60,1)}min  need={round(required_gap*60,0):.0f}min  unresolvable")
        ev_late["_ft"] = ev_early["_ft"] + MIN_CO_GAP_HRS
        break

    return adjustments

def stagger_changeovers_serial_queue(plan, machines):
    print(f"\n  Tool-Changer Serial Queue Scheduler")
    print(f"  Guarantee: strict serial — no two COs can overlap by construction")

    events = _collect_co_events(plan, machines)
    if not events:
        print(f"  No changeovers in plan — tool changer idle  ✓")
        return

    events.sort(key=lambda e: e["natural_start"])
    print(f"  {len(events)} CO events queued across {len({e['machine'] for e in events})} machines")
    print()
    print(f"  {'#':<4} {'Machine':<18} {'Part Before':<22} {'Part After':<22} "
          f"{'Dur':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7} {'Fill':>6} {'Purge':>6}")
    print(f"  {'─'*105}")

    tool_changer_free_at = 0.0
    total_extra_pcs      = 0
    total_wait_min       = 0.0

    for idx, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = round(actual_start - natural_start, 4)
        tool_changer_free_at = actual_start + co_h

        extra_pcs = 0
        if wait_hrs > 0.001:
            _, extra_pcs = _extend_row_before(ev, wait_hrs, plan)
            total_extra_pcs += extra_pcs
            total_wait_min  += wait_hrs * 60

        ev["actual_start"] = actual_start
        ev["wait_hrs"]     = wait_hrs

        wait_str  = f"+{round(wait_hrs*60,1)}m" if wait_hrs > 0.001 else "none"
        fill_str  = f"+{extra_pcs:.0f}" if extra_pcs > 0 else "—"

        before_color = part_color.get(ev["part_before"], "?")
        after_color  = part_color.get(ev["part_after"],  "?")
        purge_str    = "PURGE" if before_color != after_color and before_color not in ("?","UNKNOWN") and after_color not in ("?","UNKNOWN") else "—"

        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<22} "
              f"{ev['part_after']:<22} "
              f"{round(co_h*60,1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7} "
              f"{fill_str:>6} "
              f"{purge_str:>6}")

    n_waited = sum(1 for e in events if e.get("wait_hrs", 0) > 0.001)
    print(f"\n  Queue complete. Tool changer free at: {_fmt_h(tool_changer_free_at)}")
    print(f"  Events that had to wait : {n_waited} / {len(events)}")
    print(f"  Total wait time filled  : {round(total_wait_min, 1)} min")
    print(f"  Extra pieces produced   : {total_extra_pcs:,.0f}")
    print(f"  Overlap guarantee       : ABSOLUTE — serial queue")

def stagger_changeovers(plan, machines):
    stagger_changeovers_serial_queue(plan, machines)


# =============================================================
# 22H UTILIZATION ENFORCER  — CHANGE 4 applied throughout
#
# CHANGE 4 summary for the enforcer:
#   • Step 0 (floor extension): target = 90% (not 100%).
#     If a machine is below 90% we extend runs to reach 90%.
#     We do NOT push beyond 90% in Step 0.
#   • Steps 1-4: all "free time" calculations use the 90% soft
#     cap, so new parts are only added within 0-90% headroom.
#   • Machines intentionally left at 90% provide a 10% buffer
#     (~2.2 h) for tool/machine breakdown recovery.
# =============================================================

def _add_part_to_machine(p, m, run_hrs, co_hrs, machine_hours,
                          machine_last_part, current_inventory,
                          already_planned, plan, priority_scores,
                          type_label, role_label):
    r_val = rate.get(p, 1)
    qty   = round(run_hrs * r_val, 0)

    last      = machine_last_part.get(m)
    p_color   = part_color.get(p,    "UNKNOWN")
    l_color   = part_color.get(last, "UNKNOWN") if last else "UNKNOWN"
    has_purge = (last is not None and last != p
                 and p_color != l_color
                 and p_color != "UNKNOWN" and l_color != "UNKNOWN")

    machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
    current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
    machine_last_part[m] = p
    already_planned.add(p)

    # Consume terminals
    _consume_terminals(p, qty)

    plan.append({
        "Part":             p,
        "Color":            p_color,
        "Category":         part_category.get(p, "Stranger"),
        "Machine":          m,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
        "Daily_Indent":     round(indent_daily.get(p, 0), 2),
        "Today_Target":     round(today_target_qty.get(p, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Color_Purge":      "Yes" if has_purge else "No",
        "Type":             type_label,
        "Role":             role_label,
        "Tools_Available":  tools_available.get(p, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   round(priority_scores.get(p, 0), 2),
        "Phase":            1,
        "Stagger_Adjusted": "No",
        "Inv_Scenario":     _inv_scenario_label(p, inventory),
        "Required_Terminals": terminals_for_display(p),
    })
    return qty


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores):
    """
    CHANGE 4 — 90% soft ceiling throughout.

    Step 0 — FLOOR EXTENSION to 90%:
      Machines below 90% get their existing part runs extended
      until the machine hits exactly 90%.  No new changeovers.
      Uncapped with respect to inventory (breakdown buffer takes
      priority over OPD cap in Step 0).

    Steps 1-4 — all use soft_cap_hrs (90%) as the upper bound.
      Only Step 0 touches the 90-100% band; normal scheduling
      never enters it.

    CHANGE 2 & 3 for enforcer Steps 2-4:
      Repeater/Stranger parts: added only if NOT already planned
      (Step 2), and run on ONE machine only (no re-run via Step 3).
    """
    soft_cap_hrs = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)   # 19.8 h @ 90%

    print(f"\n  22H UTILIZATION ENFORCER  (soft ceiling = {UTIL_TARGET_PCT}% = {soft_cap_hrs:.1f}h)")
    print(f"  Step 0 fires for machines below {UTIL_TARGET_PCT}% — extends to exactly 90%")
    print(f"  Remaining 10% ({AVAILABLE_HOURS - soft_cap_hrs:.1f}h) reserved for breakdown buffer")
    micro_idle_log = []

    all_skipped = [
        p for p in all_parts
        if should_skip(p)[0] and rate.get(p, 0) > 0 and indent_monthly.get(p, 0) > 0
    ]

    machines_by_util = sorted(vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        # Free time relative to 90% cap
        remaining = round(soft_cap_hrs - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        # ── STEP 0: Extend to 90% floor (uncapped inv, no new CO) ──
        current_util_hrs = machine_hours.get(m, 0)
        if current_util_hrs < soft_cap_hrs:
            needed = round(soft_cap_hrs - current_util_hrs, 4)
            parts_on_m = [row for row in plan if row["Machine"] == m]
            if parts_on_m:
                parts_on_m_sorted = sorted(
                    parts_on_m,
                    key=lambda r: priority_scores.get(r["Part"], 0),
                    reverse=True
                )
                for row in parts_on_m_sorted:
                    if needed <= 0.001:
                        break
                    p_ext   = row["Part"]
                    r_ext   = rate.get(p_ext, 1)
                    ext_hrs = min(needed, remaining)
                    if ext_hrs < 0.001:
                        continue
                    extra_qty = round(ext_hrs * r_ext, 0)

                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Floor90"

                    machine_hours[m]        = round(machine_hours.get(m, 0) + ext_hrs, 4)
                    current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
                    remaining               = round(remaining - ext_hrs, 4)
                    needed                  = round(needed - ext_hrs, 4)

                    new_util = round(machine_hours.get(m, 0) / AVAILABLE_HOURS * 100, 1)
                    print(f"    [S0-FLOOR90] {p_ext:28s} on {m:15s}  "
                          f"+{ext_hrs:.2f}h  qty+={extra_qty:.0f}  "
                          f"util now {new_util}%  [90% floor, breakdown buffer preserved]")

        if remaining < 0.05:
            continue

        # ── STEP 1: Extend existing parts (per-part OPD cap) ────
        parts_on_machine = list({row["Part"] for row in plan if row["Machine"] == m})
        for p in sorted(parts_on_machine, key=lambda x: priority_scores.get(x, 0), reverse=True):
            if remaining < 0.05:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            # CHANGE 6: per-part cap
            headroom = max(0.0, part_opd_cap(p, current_inventory) * daily_p - inv_now)
            ext_hrs  = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext_hrs < 0.05:
                continue
            extra_qty = round(ext_hrs * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Extended"
                    break
            machine_hours[m]     = round(machine_hours.get(m, 0) + ext_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + extra_qty, 0)
            remaining            = round(remaining - ext_hrs, 4)
            print(f"    [S1-EXTEND]  {p:28s} on {m:15s}  +{ext_hrs:.2f}h  qty+={extra_qty:.0f}")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 2: Unplanned compatible parts — colour-aware sort ──
        unplanned = [
            p for p in all_parts
            if p not in already_planned
            and m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
            and rate.get(p, 0) > 0
            and current_inventory.get(p, 0) < part_opd_cap(p, current_inventory) * indent_daily.get(p, 0)
        ]

        last_on_m  = machine_last_part.get(m)
        last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"

        def _co_sort_key(p):
            needs_co    = 0 if (last_on_m is None or last_on_m == p) else 1
            p_col       = part_color.get(p, "UNKNOWN")
            same_color  = (0 if (needs_co == 1 and p_col == last_color
                                 and p_col != "UNKNOWN") else 1)
            inv_now     = current_inventory.get(p, 0)
            is_critical = 1 if inv_now == 0 else 0
            sc          = priority_scores.get(p, 0)
            return (needs_co, same_color, -is_critical, -sc)

        unplanned.sort(key=_co_sort_key)

        for p in unplanned:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_qty  = part_opd_cap(p, current_inventory) * daily_p   # CHANGE 6
            headroom = max(0.0, cap_qty - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            run_hrs   = min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            p_color  = part_color.get(p, "UNKNOWN")
            co_tag   = "No CO" if co_hrs == 0 else (
                       f"CO+PURGE({p_color})" if co_hrs > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                       else f"CO({p_color})")

            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Unplanned", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S2-UNPLAN]  {p:28s} → {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  [{co_tag}]")

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 3: Re-run planned parts (spare tool) ─────────────
        # CHANGE 2: Only Runner parts get re-run on a second machine via spare tool.
        # Repeater/Stranger must NOT be added to a second machine here.
        planned_parts_elsewhere = [
            p for p in already_planned
            if part_category.get(p, "Stranger") == "Runner"   # CHANGE 2
            and m in vt_compat.get(p, [])
            and m not in [row["Machine"] for row in plan if row["Part"] == p]
            and tools_available.get(p, 1) > len({row["Machine"] for row in plan if row["Part"] == p})
        ]
        planned_parts_elsewhere.sort(key=lambda p: (
            0 if (last_on_m is None or last_on_m == p) else 1,
            0 if (part_color.get(p, "UNKNOWN") == last_color and last_color != "UNKNOWN") else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,
            -priority_scores.get(p, 0)
        ))

        for p in planned_parts_elsewhere:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_qty  = part_opd_cap(p, current_inventory) * daily_p   # CHANGE 6
            headroom = max(0.0, cap_qty - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            run_hrs   = min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free))
            run_hrs   = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            qty = round(run_hrs * rate.get(p, 1), 0)
            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            remaining            = round(remaining - co_hrs - run_hrs, 4)

            p_color       = part_color.get(p, "UNKNOWN")
            last_p        = machine_state.get(m)
            lc            = part_color.get(last_p, "UNKNOWN") if last_p else "UNKNOWN"
            has_purge_s3  = (last_p is not None and last_p != p
                             and p_color != lc and p_color != "UNKNOWN" and lc != "UNKNOWN")

            tools_used_now = len({row["Machine"] for row in plan if row["Part"] == p}) + 1
            plan.append({
                "Part":             p,
                "Color":            p_color,
                "Category":         part_category.get(p, "Stranger"),
                "Machine":          m,
                "Run_Hours":        round(run_hrs, 3),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour":    round(rate.get(p, 1), 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(daily_p, 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Color_Purge":      "Yes" if has_purge_s3 else "No",
                "Type":             "Re-run (spare tool) [Runner only]",
                "Role":             f"Tool-Expansion (tool {tools_used_now})",
                "Tools_Available":  tools_available.get(p, 1),
                "Tools_Used":       tools_used_now,
                "Runner_Lock":      "No",
                "Priority_Score":   round(priority_scores.get(p, 0), 2),
                "Phase":            3,
                "Stagger_Adjusted": "No",
                "Inv_Scenario":     _inv_scenario_label(p, inventory),
                "Required_Terminals": terminals_for_display(p),
            })
            print(f"    [S3-RERUN]   {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  color={p_color}  tool {tools_used_now}/{tools_available.get(p,1)}")
            break

        if remaining < MIN_RUN_HOURS:
            continue

        # ── STEP 4: Skipped parts (last resort) ───────────────────
        skipped_candidates = [
            p for p in all_skipped
            if m in vt_compat.get(p, []) and rate.get(p, 0) > 0
        ]
        skipped_candidates.sort(key=lambda p: (
            0 if (last_on_m is None or last_on_m == p) else 1,
            0 if (part_color.get(p, "UNKNOWN") == last_color and last_color != "UNKNOWN") else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,
            -indent_daily.get(p, 0)
        ))

        for p in skipped_candidates:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            r_val   = rate.get(p, 1)
            run_hrs = min(eff_free, max(MIN_RUN_HOURS,
                          indent_monthly.get(p, 0) / r_val if r_val > 0 else eff_free))
            run_hrs = max(MIN_RUN_HOURS, min(run_hrs, eff_free))

            qty = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Skipped (last resort)", "Primary")
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S4-SKIPPED] {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  color={part_color.get(p,'?')}")
            break

        # ── STEP 5: Log micro-idle ────────────────────────────────
        final_util_hrs = machine_hours.get(m, 0)
        final_remaining = round(soft_cap_hrs - final_util_hrs, 4)
        util_final = round(final_util_hrs / AVAILABLE_HOURS * 100, 1)
        if util_final < UTIL_TARGET_PCT and final_remaining >= 0.25:
            micro_idle_log.append({
                "Machine":         m,
                "Scheduled_Hrs":   round(final_util_hrs, 3),
                "Soft_Cap_Hrs":    soft_cap_hrs,
                "Idle_Hrs":        round(final_remaining, 3),
                "Utilization_Pct": util_final,
                "Breakdown_Buffer_Hrs": round(AVAILABLE_HOURS - final_util_hrs, 3),
                "Note":            "All options exhausted — could not reach 90% floor",
            })
            print(f"    [⚠ IDLE]     {m:15s}  {final_remaining:.2f}h below 90% cap  ({util_final}%)")
        elif util_final >= UTIL_TARGET_PCT:
            breakdown_buf = round(AVAILABLE_HOURS - final_util_hrs, 2)
            print(f"    [✓ 90%]      {m:15s}  {util_final}%  breakdown buffer: {breakdown_buf:.2f}h")

    return micro_idle_log


# =============================================================
# OUTPUT VIEW BUILDERS
# =============================================================

def build_multi_machine_view(plan):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}
    if not multi:
        return pd.DataFrame()

    output_rows = []
    for part, rows in sorted(multi.items(), key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily     = indent_daily.get(part, 0)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)
        for row in rows:
            output_rows.append({
                "Part":                   part,
                "Color":                  part_color.get(part, "UNKNOWN"),
                "Category":               part_category.get(part, "Stranger"),
                "Tools_Available":        tools_available.get(part, 1),
                "Machines_Used":          len(rows),
                "Machine":                row["Machine"],
                "Role":                   row.get("Role", "Primary"),
                "Run_Hours":              round(float(row["Run_Hours"]), 2),
                "Changeover_Hrs":         round(float(row.get("Changeover_Hrs", 0)), 2),
                "Color_Purge":            row.get("Color_Purge", "No"),
                "Production_Qty":         round(float(row["Production_Qty"]), 0),
                "Daily_Indent":           round(daily, 2),
                "Total_Qty_All_Machines": round(total_qty, 0),
                "Type":                   row.get("Type", "—"),
            })
        output_rows.append({
            "Part":                   f"  ↳ TOTAL — {part}",
            "Color":                  part_color.get(part, "UNKNOWN"),
            "Category":               "—",
            "Tools_Available":        tools_available.get(part, 1),
            "Machines_Used":          len(rows),
            "Machine":                f"{len(rows)} machines",
            "Role":                   "TOTAL",
            "Run_Hours":              round(sum(float(r["Run_Hours"]) for r in rows), 2),
            "Changeover_Hrs":         round(sum(float(r.get("Changeover_Hrs", 0)) for r in rows), 2),
            "Color_Purge":            "—",
            "Production_Qty":         round(total_qty, 0),
            "Daily_Indent":           round(daily, 2),
            "Total_Qty_All_Machines": round(total_qty, 0),
            "Type":                   "—",
        })
        output_rows.append({k: "" for k in output_rows[-1].keys()})
    return pd.DataFrame(output_rows)


def build_production_vs_indent(plan, all_parts):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])

    rows = []
    for p in sorted(part_qty.keys()):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap      = round(produced - daily, 0)
        gap_dir  = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")
        extra_days = round(gap / daily, 2) if daily > 0 and gap > 0 else 0.0
        rows.append({
            "Part":               p,
            "Color":              part_color.get(p, "UNKNOWN"),
            "Category":           part_category.get(p, "Stranger"),
            "Tools_Available":    tools_available.get(p, 1),
            "Inv_Scenario":       _inv_scenario_label(p, inventory),
            "Machines":           ", ".join(dict.fromkeys(part_machines[p])),
            "Machines_Count":     len(set(part_machines[p])),
            "Total_Qty_Produced": produced,
            "Daily_Indent":       round(daily, 2),
            "Monthly_Indent":     round(monthly, 0),
            "Gap_vs_Daily":       gap,
            "Gap_Direction":      gap_dir,
            "Extra_Days_Stock":   extra_days,
            "Inventory_Before":   round(inv_b, 0),
            "Inventory_After":    inv_after,
            "Days_Coverage_After":round(inv_after / daily, 2) if daily > 0 else 0,
        })

    order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Daily"]).drop(columns=["_sort"]).reset_index(drop=True)
    return df


def build_inventory_target_sheet(plan, all_parts, scenario_id):
    """
    CHANGE 6: Buffer_Status and OPD_Cap now use the S0/S1/S2/S3
    framework from the slide (per-part assessment).
    """
    from collections import defaultdict
    part_produced = defaultdict(float)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))

    rows = []
    for p in sorted(all_parts):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        days_before = round(inv_b    / daily, 2) if daily > 0 else 0
        days_after  = round(inv_after / daily, 2) if daily > 0 else 0
        target_qty  = round(TARGET_DAYS * daily, 0)
        gap_qty     = round(target_qty - inv_after, 0)
        gap_days    = max(0, round(gap_qty / daily, 2) if daily > 0 else 0)

        # Per-part OPD cap (CHANGE 6)
        cap         = part_opd_cap(p)   # based on opening inventory
        max_prod    = round(cap * daily, 0)

        # S0/S1/S2/S3 scenario label (CHANGE 6)
        inv_scenario = _inv_scenario_label(p, inventory)

        if inv_after == 0:
            status = "CRITICAL"
        elif days_after < SAFETY_DAYS:
            status = "BELOW_SAFETY"
        elif days_after < TARGET_DAYS:
            status = "BUILDING"
        else:
            status = "AT_TARGET"

        net_gain = (cap - 1) * daily if daily > 0 else 0
        if gap_qty <= 0:
            est_days = "AT TARGET"
        elif net_gain <= 0:
            est_days = "N/A"
        else:
            est_days = str(math.ceil(gap_qty / net_gain)) + " days"

        skip, skip_reason = should_skip(p)
        rows.append({
            "Part":                   p,
            "Color":                  part_color.get(p, "UNKNOWN"),
            "Category":               part_category.get(p, "Stranger"),
            "Tools":                  tools_available.get(p, 1),
            "Inv_Scenario":           inv_scenario,       # CHANGE 6
            "Monthly_Indent":         round(monthly, 0),
            "Daily_Indent":           round(daily, 2),
            "Target_Qty_5days":       target_qty,
            "Safety_Floor_Qty_3days": round(SAFETY_DAYS * daily, 0),
            "Inv_Before":             round(inv_b, 0),
            "Days_Coverage_Before":   days_before,
            "Produced_Today":         produced,
            "Inv_After":              inv_after,
            "Days_Coverage_After":    days_after,
            "Gap_to_Target_Qty":      max(0, gap_qty),
            "Gap_to_Target_Days":     gap_days,
            "Buffer_Status":          status,
            "OPD_Cap_Today_Days":     cap,               # CHANGE 6 (per-part)
            "Max_Producible_Qty":     max_prod,
            "Est_Days_to_Target":     est_days,
            "Scheduled_Today":        ("YES" if produced > 0
                                       else "SKIPPED — AT TARGET" if inv_b >= target_qty
                                       else "NO — no capacity"),
            "Skip_Reason":            skip_reason if skip else "",
        })

    status_order = {"CRITICAL": 0, "BELOW_SAFETY": 1, "BUILDING": 2, "AT_TARGET": 3}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Buffer_Status"].map(status_order)
        df = df.sort_values(["_sort", "Gap_to_Target_Days"], ascending=[True, False])
        df = df.drop(columns=["_sort"]).reset_index(drop=True)
    return df


def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0
        days_cov   = inv / daily if daily > 0 else 0

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip and inv >= TARGET_DAYS * daily:
            status = "AT 5-DAY TARGET — SKIP"
        elif skip and part_category.get(p, "Stranger") == "Runner":
            status = "RUNNER DEFERRED (≥3 day buffer)"   # CHANGE 5
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":             p,
            "Color":            part_color.get(p, "UNKNOWN"),
            "Category":         part_category.get(p, "Stranger"),
            "Tools":            tools_available.get(p, 1),
            "Inv_Scenario":     _inv_scenario_label(p, inventory),   # CHANGE 6
            "Monthly_Indent":   round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Working_Days":     WORKING_DAYS,
            "Daily_Indent":     round(daily, 2),
            "Inventory_Now":    round(inv, 0),
            "Days_Coverage":    round(days_cov, 2),
            "Safety_Floor":     SAFETY_DAYS,
            "Target_Ceiling":   TARGET_DAYS,
            "Today_Target_Qty": round(target, 0),
            "Today_Target_Hrs": round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour":    round(r, 2),
            "Indent_Status":    status,
            "Skip_Reason":      skip_reason,
        })
    return pd.DataFrame(rows)


SPECIALIZED_MACHINE_THRESHOLD = 3

def detect_specialized_machines(all_parts):
    machine_parts = {}
    for m in vt_machines:
        machine_parts[m] = [
            p for p in all_parts
            if m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
        ]
    specialized_machines = {m for m, mp in machine_parts.items() if 0 < len(mp) <= SPECIALIZED_MACHINE_THRESHOLD}
    return specialized_machines, {m: machine_parts[m] for m in specialized_machines}, machine_parts


# =============================================================
# MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):
    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"  Safety floor: {SAFETY_DAYS} days  |  Target ceiling: {TARGET_DAYS} days")
    print(f"  Util soft ceiling: {UTIL_TARGET_PCT}%  |  Breakdown buffer: {100-UTIL_TARGET_PCT}%")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")
    print(f"  Global OPD cap today: {opd_cap(scenario_id)} days  "
          f"(per-part caps vary by S0-S3 inventory scenario)")

    horizon_df = compute_indent_horizon(parts)

    active_parts = [
        p for p in parts
        if not should_skip(p)[0] and indent_monthly.get(p, 0) > 0
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    machine_hours     = {m: 0.0 for m in vt_machines}
    machine_last_part = {m: machine_state.get(m) for m in vt_machines}
    current_inventory = inventory.copy()
    plan              = []
    already_planned   = set()
    not_planned       = []
    deferred          = []
    displaced_log     = []

    specialized_machines, spec_machine_parts, _ = detect_specialized_machines(list(parts))
    print(f"\n  SPECIALIZED MACHINES (≤{SPECIALIZED_MACHINE_THRESHOLD} parts):")
    if specialized_machines:
        for m in sorted(specialized_machines, key=lambda m: machine_part_count.get(m, 99)):
            mparts = spec_machine_parts.get(m, [])
            cnt    = machine_part_count.get(m, "?")
            print(f"    {m:<25} Part_Count={cnt:<4} parts: {', '.join(mparts)}")
    else:
        print(f"    None — all machines have >{SPECIALIZED_MACHINE_THRESHOLD} compatible parts")

    zero_inv_rr = [
        p for p in active_parts
        if current_inventory.get(p, 0) == 0
        and part_category.get(p, "Stranger") in ("Runner", "Repeater")
        and vt_compat.get(p)
    ]

    if zero_inv_rr:
        print(f"\n  V9 DISPLACEMENT PRE-PASS  ({len(zero_inv_rr)} zero-inv Runner/Repeater parts)")
    else:
        print(f"\n  V9 DISPLACEMENT PRE-PASS  — no zero-inv Runner/Repeater parts today  ✓")

    sorted_active = sorted(active_parts, key=lambda p: priority_scores.get(p, 0), reverse=True)

    print(f"\n  PRIMARY SCHEDULING PASS  ({len(sorted_active)} active parts)")
    print(f"  {'Part':<30} {'Cat':<10} {'Color':<10} {'Score':>6} {'Days':>5} "
          f"{'Scenario':<16} {'Status':<15} {'Machine(s)':<25} {'Run':>5} {'Qty':>8}")
    print(f"  {'─'*140}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        score    = priority_scores.get(part, 0)
        tools    = tools_available.get(part, 1)
        category = part_category.get(part, "Stranger")
        color    = part_color.get(part, "UNKNOWN")
        days_cov = inv_now / daily if daily > 0 else 999
        inv_scen = _inv_scenario_label(part, inventory)

        buf_label = ("CRITICAL"     if inv_now == 0 else
                     "BELOW_SAFETY" if days_cov < SAFETY_DAYS else
                     "BUILDING"     if days_cov < TARGET_DAYS else
                     "AT_TARGET")

        if monthly == 0:
            deferred.append({"Part": part, "Color": color, "Category": category, "Reason": "Monthly indent = 0"})
            print(f"  {part:<30} {category:<10} {color:<10} {score:>6.1f} {days_cov:>5.1f} "
                  f"{inv_scen:<16} {'DEFERRED':<15}  —  no indent")
            continue

        if not vt_compat.get(part):
            not_planned.append({
                "Part": part, "Color": color, "Category": category, "Score": score,
                "Daily_Indent": round(daily, 2), "Inventory_Now": round(inv_now, 0),
                "Tools": tools, "Compatible_Machines": "NONE DEFINED",
                "Reason": "Not in VT_Matrix", "Action_Needed": "Add to VT_Matrix",
            })
            print(f"  {part:<30} {category:<10} {color:<10} {score:>6.1f} {days_cov:>5.1f} "
                  f"{inv_scen:<16} {buf_label:<15}  ✗ NOT IN MATRIX")
            continue

        new_rows = assign_part(part, scenario_id, machine_hours, machine_last_part,
                               current_inventory, plan, already_planned, priority_scores)

        if new_rows:
            plan.extend(new_rows)
            machines_used = [r["Machine"] for r in new_rows]
            total_qty     = sum(float(r["Production_Qty"]) for r in new_rows)
            total_run     = sum(float(r["Run_Hours"]) for r in new_rows)
            machines_str  = ", ".join(machines_used)
            flag = " [MULTI]" if len(new_rows) > 1 else ""
            flag += " [ZERO-INV]" if inv_now == 0 else ""
            purges = sum(1 for r in new_rows if r.get("Color_Purge") == "Yes")
            flag  += f" [PURGE×{purges}]" if purges > 0 else ""
            if category in ("Repeater", "Stranger") and len(new_rows) > 1:
                flag += " [WARN:MULTI-MACHINE for R/S]"
            print(f"  {part:<30} {category:<10} {color:<10} {score:>6.1f} {days_cov:>5.1f} "
                  f"{inv_scen:<16} {buf_label:<15}  "
                  f"{machines_str:<25}  {total_run:>5.2f}  {total_qty:>8.0f}  ✓{flag}")
        else:
            not_planned.append({
                "Part":                part,
                "Color":               color,
                "Category":            category,
                "Score":               score,
                "Tools":               tools,
                "Days_Coverage":       round(days_cov, 2),
                "Buffer_Status":       buf_label,
                "Daily_Indent":        round(daily, 2),
                "Inventory_Now":       round(inv_now, 0),
                "Compatible_Machines": ", ".join(vt_compat.get(part, [])),
                "Reason":              "No compatible machine has capacity",
                "Action_Needed":       "Review matrix or add machines",
            })
            print(f"  {part:<30} {category:<10} {color:<10} {score:>6.1f} {days_cov:>5.1f} "
                  f"{inv_scen:<16} {buf_label:<15}  ✗ NO CAPACITY")

    # Post-primary displacement
    unscheduled_zero_rr = [p for p in zero_inv_rr if p not in already_planned]
    if unscheduled_zero_rr:
        print(f"\n  V9 DISPLACEMENT PASS  ({len(unscheduled_zero_rr)} zero-inv R/R unscheduled)")
        for part in unscheduled_zero_rr:
            success = displace_for_zero_inv(
                part, machine_hours, machine_last_part,
                current_inventory, plan, already_planned, priority_scores)
            if success:
                displaced_log.append(part)
                not_planned[:] = [r for r in not_planned if r.get("Part") != part]
            else:
                print(f"      ↳ DISPLACEMENT FAILED  {part}  — no suitable victim found")
    else:
        print(f"\n  V9 DISPLACEMENT PASS  — no unscheduled zero-inv R/R parts  ✓")

    # Enforcer
    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(parts), already_planned, current_inventory, scenario_id, priority_scores)

    # CO stagger
    print(f"\n  CO Quantity Stagger  (min gap = {MIN_CO_GAP_HRS*60:.0f} min after CO finishes)")
    n_adj = stagger_co_by_quantity(plan, vt_machines, scenario_id, current_inventory)
    print(f"    {'No adjustments needed  ✓' if n_adj == 0 else str(n_adj) + ' adjustment(s) made'}")

    stagger_changeovers(plan, vt_machines)

    # Build views
    multi_machine_df  = build_multi_machine_view(plan)
    prod_vs_indent_df = build_production_vs_indent(plan, list(parts))
    inv_target_df     = build_inventory_target_sheet(plan, list(parts), scenario_id)

    # Daily indent status
    indent_status_rows = []
    for row in plan:
        p       = row["Part"]
        planned = float(row.get("Production_Qty") or 0)
        daily   = indent_daily.get(p, 0)
        inv_b   = inventory.get(p, 0)
        meets   = planned >= daily
        indent_status_rows.append({
            "Part":               p,
            "Color":              part_color.get(p, "UNKNOWN"),
            "Category":           part_category.get(p, "Stranger"),
            "Machine":            row.get("Machine", "—"),
            "Color_Purge":        row.get("Color_Purge", "No"),
            "Role":               row.get("Role", "Primary"),
            "Priority_Score":     round(row.get("Priority_Score", 0), 2),
            "Inv_Scenario":       row.get("Inv_Scenario", _inv_scenario_label(p, inventory)),
            "Buffer_Status":      (
                "CRITICAL"     if inv_b == 0 else
                "BELOW_SAFETY" if (daily > 0 and inv_b / daily < SAFETY_DAYS) else
                "BUILDING"     if (daily > 0 and inv_b / daily < TARGET_DAYS) else
                "AT_TARGET"
            ),
            "Run_Hours":          round(float(row.get("Run_Hours") or 0), 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Gap_vs_Daily":       round(daily - planned, 0),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),
            "Covers_With_Inv":    "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
        })
    indent_status_df = pd.DataFrame(indent_status_rows)
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"], ascending=[True, False]).reset_index(drop=True)

    # Inventory health
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Color":           part_color.get(p, "UNKNOWN"),
            "Category":        part_category.get(p, "Stranger"),
            "Tools":           tools_available.get(p, 1),
            "Inv_Scenario":    _inv_scenario_label(p, inventory),   # CHANGE 6
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Monthly_Indent":  round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Safety_Floor":    SAFETY_DAYS,
            "Target_Ceiling":  TARGET_DAYS,
            "Status":          (
                "AT_TARGET" if days_cov >= TARGET_DAYS else
                "OK"        if days_cov >= SAFETY_DAYS else
                "LOW"       if days_cov >= 1 else
                "CRITICAL"
            ),
        })

    # Machine utilisation — CHANGE 4: show soft cap and breakdown buffer
    mach_rows = []
    for m in vt_machines:
        used      = machine_hours.get(m, 0)
        parts_run = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count  = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        purge_count = sum(1 for r in plan if r["Machine"] == m and r.get("Color_Purge") == "Yes")
        colors_on_machine = list({part_color.get(r["Part"], "UNKNOWN")
                                   for r in plan if r["Machine"] == m})
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        soft_cap  = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)
        brkdwn_buf = round(AVAILABLE_HOURS - used, 2)

        mach_rows.append({
            "Machine":              m,
            "Colors_Today":         ", ".join(sorted(colors_on_machine)),
            "Color_Purges":         purge_count,
            "Used_Hours":           round(used, 2),
            "Soft_Cap_Hours":       round(soft_cap, 2),
            "Breakdown_Buffer_Hrs": brkdwn_buf,
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            # CHANGE 4: status now references 90% ceiling
            "Status":               ("FULL"       if used >= soft_cap - 0.1 else
                                     "GOOD"       if used >= soft_cap * 0.98 else
                                     "OK"         if used >= soft_cap * 0.90 else
                                     "PARTIAL"    if used >= soft_cap * 0.85 else
                                     "UNDERUSED"),
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": ", ".join(parts_run) if parts_run else "— idle —",
        })

    micro_df  = pd.DataFrame(micro_idle)   if micro_idle   else pd.DataFrame()
    plan_df   = pd.DataFrame(plan)         if plan         else pd.DataFrame()
    def_df    = pd.DataFrame(deferred)     if deferred     else pd.DataFrame()
    not_df    = pd.DataFrame(not_planned)  if not_planned  else pd.DataFrame()
    mach_df   = pd.DataFrame(mach_rows)
    inv_df    = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date",   str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",        scenario_desc)
        plan_df.insert(2, "Working_Days",    WORKING_DAYS)
        plan_df.insert(3, "Safety_Days",     SAFETY_DAYS)
        plan_df.insert(4, "Target_Days",     TARGET_DAYS)
        plan_df.insert(5, "OPD_Cap_Today",   opd_cap(scenario_id))
        plan_df.insert(6, "Util_Soft_Cap_%", UTIL_TARGET_PCT)

    n_displaced = len(displaced_log)
    print(f"\n  {'='*65}")
    print(f"  SCHEDULE SUMMARY — {scenario_desc}")
    print(f"    Parts planned           : {len(already_planned)}")
    print(f"    Displaced (zero-inv R/R): {n_displaced}")
    print(f"    Not planned             : {len(not_planned)}")
    print(f"    Deferred                : {len(deferred)}")
    if not mach_df.empty:
        print(f"    Avg utilization         : {mach_df['Utilization_%'].mean():.1f}%")
        print(f"    Avg breakdown buffer    : {mach_df['Breakdown_Buffer_Hrs'].mean():.2f}h per machine")
        total_purges = mach_df["Color_Purges"].sum()
        print(f"    Total colour purges     : {total_purges}")
    if not inv_target_df.empty:
        at_t = (inv_target_df["Buffer_Status"] == "AT_TARGET").sum()
        bld  = (inv_target_df["Buffer_Status"] == "BUILDING").sum()
        bls  = (inv_target_df["Buffer_Status"] == "BELOW_SAFETY").sum()
        crt  = (inv_target_df["Buffer_Status"] == "CRITICAL").sum()
        print(f"\n    Inventory target status (after today):")
        print(f"      S3-HEALTHY  AT_TARGET   (≥5 days) : {at_t}")
        print(f"      S2-BELOW    BUILDING  (3–5 days)  : {bld}")
        print(f"      S1-MIXED    BELOW_SAFETY (<3 days): {bls}")
        print(f"      S0-CRITICAL CRITICAL  (0 pcs)     : {crt}")
    print(f"  {'='*65}")

    return (plan_df, def_df, not_df, mach_df, inv_df,
            machine_last_part, horizon_df, indent_status_df,
            score_df, micro_df, multi_machine_df, prod_vs_indent_df,
            inv_target_df)


# =============================================================
# PART AUDIT
# =============================================================

vt_parts = data_valid[data_valid["Material"].isin(vt_matrix["Part"])]["Material"].unique()
all_vt_parts_raw = list(data["Material"].unique())
matrix_parts     = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set    = set(data_zero_rate["Material"].unique())

audit_rows = []
for part in all_vt_parts_raw:
    inv      = inventory.get(part, 0.0)
    r_val    = rate.get(part, None)
    monthly  = indent_monthly.get(part, 0.0)
    daily    = indent_daily.get(part, 0.0)
    row_data = data[data["Material"] == part]
    ct_raw   = row_data[vt_col_cycletime].values[0] if len(row_data) else "—"
    cv_raw   = row_data[vt_col_cavity].values[0]    if len(row_data) else "—"
    tools    = tools_available.get(part, 1)
    color    = part_color.get(part, "UNKNOWN")
    category = part_category.get(part, "Stranger")
    days_cov = inv / daily if daily > 0 else 0
    req_terms   = terminals_for_display(part)
    blk_terms   = blocked_terminals_for_display(part)
    term_blocked, term_reason = is_terminal_blocked(part)

    if part in zero_rate_set or r_val is None:
        status, gate, reason = "ZERO/MISSING CYCLE TIME", "GATE 1", f"Cycle time={ct_raw}"
    elif part not in matrix_parts:
        status, gate, reason = "NOT IN VT_MATRIX", "GATE 2", "No compatible machine"
    elif monthly == 0:
        status, gate, reason = "ZERO/MISSING INDENT", "GATE 3", "Monthly indent = 0"
    elif daily <= MIN_DAILY_INDENT:
        status, gate, reason = "SKIPPED (LOW INDENT ≤150/day)", "GATE 4a", f"Daily {daily:.2f} ≤ {MIN_DAILY_INDENT}"
    elif (monthly / r_val if r_val else 0) <= MIN_INDENT_HOURS:
        status, gate, reason = "SKIPPED (MONTHLY RUN ≤4h)", "GATE 4b", f"Monthly hrs ≤ {MIN_INDENT_HOURS}h"
    elif term_blocked:
        # TERMINAL GATE (NEW) — sits between indent thresholds and inventory gates
        status, gate, reason = "TERMINAL BLOCKED", "GATE 4c", term_reason
    elif category == "Runner" and daily > 0 and inv >= RUNNER_DEFER_INV_DAYS * daily and inv < TARGET_DAYS * daily:
        status, gate, reason = "RUNNER DEFERRED (≥3d buffer)", "GATE 5-RUNNER", \
            f"Runner: inv ({inv:.0f}) ≥ {RUNNER_DEFER_INV_DAYS}×daily. Plan tomorrow."
    elif daily > 0 and inv >= TARGET_DAYS * daily:
        status, gate, reason = f"AT {TARGET_DAYS}-DAY TARGET — SKIP TODAY", "GATE 6", \
            f"Inv ({inv:.0f}) ≥ {TARGET_DAYS}×daily ({TARGET_DAYS*daily:.0f})"
    else:
        status, gate, reason = "ENTERS SCHEDULER", "—", "Passed all gates"

    audit_rows.append({
        "Part":                part,
        "Color":               color,
        "Category":            category,
        "Gate_Failed":         gate,
        "Reason":              reason,
        "Required_Terminals":  req_terms,
        "Blocked_Terminals":   blk_terms,
        "Terminal_Status":     "BLOCKED" if term_blocked else ("OK" if req_terms != "—" else "NO DATA"),
        "Monthly_Indent":      round(monthly, 0),
        "Daily_Indent":        round(daily, 2),
        "Inventory":           round(inv, 0),
        "Days_Coverage":       round(days_cov, 2),
        "Safety_Floor":        SAFETY_DAYS,
        "Target_Ceiling":      TARGET_DAYS,
        "Tools":               tools,
        "Rate_Per_Hour":       round(r_val, 2) if r_val else "—",
        "Cycle_Time":          ct_raw,
        "Cavity":              cv_raw,
        "Status":              status,
        "Inv_Scenario":        _inv_scenario_label(part, inventory),
    })

audit_df = pd.DataFrame(audit_rows)
gate_counts = audit_df["Status"].value_counts()
print(f"\n  Part audit ({len(all_vt_parts_raw)} total):")
for status, count in gate_counts.items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"    {marker}  {status:<60}: {count:>4}")


# =============================================================
# RUN
# =============================================================

(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_indent_status,
 vt_scores, vt_micro, vt_multi_machine,
 vt_prod_vs_indent, vt_inv_target) = schedule(vt_parts, "VT Machines")

save_machine_state(vt_state)


# =============================================================
# MACHINE-WISE PLAN
# =============================================================

def build_machine_wise_plan(plan_df):
    if plan_df.empty:
        return pd.DataFrame()

    def sf(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue

        for _, pr in machine_rows.iterrows():
            p    = pr.get("Part", "—")
            co_h = sf(pr.get("Changeover_Hrs", 0))
            run_h = sf(pr.get("Run_Hours", 0))
            r_val = sf(pr.get("Rate_Per_Hour", 0))
            rows.append({
                "Machine":            m,
                "Part":               p,
                "Color":              part_color.get(p, "UNKNOWN"),
                "Category":           part_category.get(p, "Stranger"),
                "Role":               pr.get("Role", "Primary"),
                "Inv_Scenario":       pr.get("Inv_Scenario", _inv_scenario_label(p, inventory)),
                "Tools_Available":    pr.get("Tools_Available", 1),
                "Priority_Score":     round(sf(pr.get("Priority_Score", 0)), 2),
                "Rate_Per_Hour":      round(r_val, 2),
                "Run_Hours":          round(run_h, 2),
                "Changeover_Hrs":     round(co_h, 3),
                "Changeover_Needed":  pr.get("Changeover", "No") or "No",
                "Color_Purge":        pr.get("Color_Purge", "No") or "No",
                "Production_Qty":     round(sf(pr.get("Production_Qty", 0)), 0),
                "Daily_Indent":       round(sf(pr.get("Daily_Indent", 0)), 2),
                "Today_Target":       round(sf(pr.get("Today_Target", 0)), 0),
                "Monthly_Indent":     round(sf(pr.get("Monthly_Indent", 0)), 0),
                "Type":               pr.get("Type", "Primary") or "Primary",
                "Row_Type":           "Part",
            })

        co_total    = machine_rows["Changeover_Hrs"].apply(lambda x: sf(x, 0)).sum()
        run_total   = machine_rows["Run_Hours"].apply(lambda x: sf(x, 0)).sum()
        qty_total   = machine_rows["Production_Qty"].apply(lambda x: sf(x, 0)).sum()
        co_count    = int(machine_rows["Changeover"].eq("Yes").sum())
        purge_count = int(machine_rows["Color_Purge"].eq("Yes").sum()) if "Color_Purge" in machine_rows.columns else 0
        hrs_total   = round(co_total + run_total, 2)
        colors_list = sorted({part_color.get(p, "UNKNOWN") for p in machine_rows["Part"]})
        soft_cap    = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)
        brkdwn_buf  = round(AVAILABLE_HOURS - hrs_total, 2)

        rows.append({
            "Machine":            m,
            "Part":               f"TOTAL — {m}",
            "Color":              ", ".join(colors_list),
            "Category":           "—",
            "Role":               "—",
            "Inv_Scenario":       "—",
            "Tools_Available":    "—",
            "Priority_Score":     "—",
            "Rate_Per_Hour":      "—",
            "Run_Hours":          round(run_total, 2),
            "Changeover_Hrs":     round(co_total, 2),
            "Changeover_Needed":  f"{co_count} changeover(s)",
            "Color_Purge":        f"{purge_count} purge(s)",
            "Production_Qty":     round(qty_total, 0),
            "Daily_Indent":       "—",
            "Today_Target":       "—",
            "Monthly_Indent":     "—",
            "Type":               (f"Total {hrs_total}h / {soft_cap:.1f}h (90%)  |  "
                                   f"Breakdown buf {brkdwn_buf}h  |  "
                                   f"Util {round(hrs_total / AVAILABLE_HOURS * 100, 1)}%"),
            "Row_Type":           "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)


def build_co_queue(plan, machines):
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()

    events.sort(key=lambda e: e["natural_start"])
    rows = []
    tool_changer_free_at = 0.0

    for pos, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_min      = round((actual_start - natural_start) * 60, 1)
        tool_changer_free_at = actual_start + co_h

        before_color = part_color.get(ev["part_before"], "UNKNOWN")
        after_color  = part_color.get(ev["part_after"],  "UNKNOWN")
        color_change = (before_color != after_color
                        and before_color != "UNKNOWN"
                        and after_color  != "UNKNOWN")

        note = ""
        if wait_min > 0:
            note = f"Machine may need to wait ~{wait_min}min. Continue running previous part until team arrives."
        if color_change:
            note = (note + "  " if note else "") + f"COLOUR CHANGE: {before_color} → {after_color}. 10-min purge required."

        rows.append({
            "Queue_Position":   pos,
            "Machine":          ev["machine"],
            "Part_Before":      ev["part_before"],
            "Color_Before":     before_color,
            "Part_After":       ev["part_after"],
            "Color_After":      after_color,
            "Color_Change":     "YES — PURGE" if color_change else "No",
            "CO_Duration_Min":  round(co_h * 60, 1),
            "Note":             note if note else "Tool changer available immediately. No colour change.",
        })

    return pd.DataFrame(rows)


# =============================================================
# EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":      "0D6E6E",
    "VT_CO_Queue":             "375623",
    "VT_Plan":                 "1F4E79",
    "VT_Daily_Indent_Status":  "0F4C2A",
    "VT_Multi_Machine_Parts":  "4A235A",
    "VT_Production_vs_Indent": "154360",
    "VT_Inventory_Target":     "1B4F72",
    "VT_Priority_Scores":      "2C4770",
    "VT_Machine_Util":         "375623",
    "VT_Not_Planned":          "7B2C2C",
    "VT_Deferred":             "7F6000",
    "VT_Inventory_Health":     "4A235A",
    "VT_Indent_Horizon":       "154360",
    "VT_Part_Audit":           "1C3557",
    "VT_Micro_Idle":           "5C3D2E",
    "VT_Terminal_Status":      "7B3F00",
    "VT_Manual_Intervention":  "7B0000",   # ← deep red — urgent action sheet
}

STATUS_FILLS = {
    "FULL":         PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":         PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":      PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":    PatternFill("solid", fgColor="FFC7CE"),
    "OK":           PatternFill("solid", fgColor="C6EFCE"),
    "LOW":          PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":     PatternFill("solid", fgColor="FFC7CE"),
    "AT_TARGET":    PatternFill("solid", fgColor="C6EFCE"),
    "BUILDING":     PatternFill("solid", fgColor="DDEBF7"),
    "BELOW_SAFETY": PatternFill("solid", fgColor="FFEB9C"),
    "YES ✓":        PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":         PatternFill("solid", fgColor="FFC7CE"),
    "OVER":         PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":        PatternFill("solid", fgColor="FFC7CE"),
    "MET":          PatternFill("solid", fgColor="C6EFCE"),
    "YES — PURGE":  PatternFill("solid", fgColor="FFC7CE"),
    "Yes":          PatternFill("solid", fgColor="FFEB9C"),
    # S0/S1/S2/S3 scenario fills
    "S0-CRITICAL":       PatternFill("solid", fgColor="FFC7CE"),
    "S1-MIXED_RISK":     PatternFill("solid", fgColor="FFEB9C"),
    "S2-BELOW_TARGET":   PatternFill("solid", fgColor="DDEBF7"),
    "S3-HEALTHY":        PatternFill("solid", fgColor="C6EFCE"),
    # Runner defer
    "RUNNER DEFERRED (≥3d buffer)":    PatternFill("solid", fgColor="FFF2CC"),
    "RUNNER DEFERRED (≥3 day buffer)": PatternFill("solid", fgColor="FFF2CC"),
    # Terminal status fills
    "BLOCKED":            PatternFill("solid", fgColor="FFC7CE"),
    "CLEAR":              PatternFill("solid", fgColor="C6EFCE"),
    "OPERATIONAL":        PatternFill("solid", fgColor="C6EFCE"),
    "DOWN — BLOCKING":    PatternFill("solid", fgColor="FF0000"),
    "TERMINAL BLOCKED":   PatternFill("solid", fgColor="FFC7CE"),
    "NO DATA":            PatternFill("solid", fgColor="EDEDED"),
    # Terminal criticality fills (for Terminal_Criticality column)
    "HIGHLY CRITICAL":    PatternFill("solid", fgColor="FF0000"),   # bright red
    "CRITICAL":           PatternFill("solid", fgColor="FFC7CE"),   # pink
    "OK":                 PatternFill("solid", fgColor="C6EFCE"),   # green
    "PRODUCTION NEEDED":                     PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":                     PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":                        PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":                               PatternFill("solid", fgColor="EDEDED"),
    f"AT {TARGET_DAYS}-DAY TARGET — SKIP TODAY": PatternFill("solid", fgColor="C6EFCE"),
    "NOT REQUIRED — INV SUFFICIENT":         PatternFill("solid", fgColor="DDEBF7"),
    "SKIPPED (LOW INDENT ≤150/day)":         PatternFill("solid", fgColor="EDEDED"),
    "SKIPPED (MONTHLY RUN ≤4h)":             PatternFill("solid", fgColor="EDEDED"),
    "ZERO/MISSING CYCLE TIME":               PatternFill("solid", fgColor="FFC7CE"),
    "NOT IN VT_MATRIX":                      PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":                   PatternFill("solid", fgColor="FFEB9C"),
    "ENTERS SCHEDULER":                      PatternFill("solid", fgColor="C6EFCE"),
}

def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in
                            ["Status", "Indent_Status", "Meets_Daily",
                             "Covers_With", "Gap_Direction", "Buffer_Status",
                             "Scheduled_Today", "Color_Change", "Color_Purge",
                             "Inv_Scenario", "Terminal_Status",
                             "Terminal_Criticality"]):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def style_inv_target_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Inventory_Target"])
    headers    = [c.value for c in ws[1]]
    status_col = headers.index("Buffer_Status") + 1 if "Buffer_Status" in headers else None
    row_fills  = {
        "CRITICAL":     PatternFill("solid", fgColor="FFD7D7"),
        "BELOW_SAFETY": PatternFill("solid", fgColor="FFF2CC"),
        "BUILDING":     PatternFill("solid", fgColor="DDEEFF"),
        "AT_TARGET":    PatternFill("solid", fgColor="E2EFDA"),
    }
    for row in ws.iter_rows(min_row=2):
        if not status_col:
            continue
        status = str(row[status_col - 1].value)
        fill   = row_fills.get(status)
        if fill:
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in ("00000000", "FFFFFFFF"):
                    cell.fill = fill
        if status == "CRITICAL":
            for cell in row:
                cell.font = Font(bold=True)


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    summary_fill = PatternFill("solid", fgColor="0D9488")
    part_fills   = [PatternFill("solid", fgColor="EFF6FF"), PatternFill("solid", fgColor="F0FDF4")]
    co_fill      = PatternFill("solid", fgColor="FEF9C3")
    purge_fill   = PatternFill("solid", fgColor="FFC7CE")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers       = [cell.value for cell in ws[1]]
    row_type_col  = headers.index("Row_Type")     + 1 if "Row_Type"     in headers else None
    co_col        = headers.index("Changeover_Needed") + 1 if "Changeover_Needed" in headers else None
    purge_col     = headers.index("Color_Purge")  + 1 if "Color_Purge"  in headers else None
    machine_col   = headers.index("Machine")      + 1 if "Machine"      in headers else None

    machine_color_idx = 0
    current_machine   = None
    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col-1].value if row_type_col else ""
        machine  = row[machine_col-1].value  if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            for cell in row:
                cell.fill      = part_fills[machine_color_idx]
                cell.alignment = Alignment(vertical="center")
            if co_col and str(row[co_col-1].value) == "Yes":
                row[co_col-1].fill = co_fill
            if purge_col and str(row[purge_col-1].value) == "Yes":
                row[purge_col-1].fill = purge_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "B2"


def style_prod_vs_indent_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Production_vs_Indent"])
    headers  = [c.value for c in ws[1]]
    gap_col  = headers.index("Gap_Direction") + 1 if "Gap_Direction" in headers else None
    over_fill  = PatternFill("solid", fgColor="DDEBF7")
    under_fill = PatternFill("solid", fgColor="FFC7CE")
    met_fill   = PatternFill("solid", fgColor="C6EFCE")
    for row in ws.iter_rows(min_row=2):
        if gap_col:
            cell  = row[gap_col - 1]
            value = str(cell.value)
            if value == "OVER":
                cell.fill = over_fill
            elif value == "UNDER":
                cell.fill = under_fill
                for c in row:
                    c.font = Font(bold=True)
            elif value == "MET":
                cell.fill = met_fill


def style_multi_machine_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Multi_Machine_Parts"])
    headers  = [c.value for c in ws[1]]
    role_col = headers.index("Role") + 1 if "Role" in headers else None
    total_fill   = PatternFill("solid", fgColor="0D9488")
    primary_fill = PatternFill("solid", fgColor="EFF6FF")
    expand_fill  = PatternFill("solid", fgColor="FEF9C3")
    for row in ws.iter_rows(min_row=2):
        if not role_col:
            continue
        role = str(row[role_col - 1].value)
        if role == "TOTAL":
            for cell in row:
                cell.fill = total_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif "Tool-Expansion" in role:
            for cell in row:
                cell.fill = expand_fill
        elif role == "Primary":
            for cell in row:
                cell.fill = primary_fill


def style_co_queue_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_CO_Queue"])
    headers      = [c.value for c in ws[1]]
    cc_col       = headers.index("Color_Change") + 1 if "Color_Change" in headers else None
    purge_fill   = PatternFill("solid", fgColor="FFC7CE")
    for row in ws.iter_rows(min_row=2):
        if cc_col and str(row[cc_col-1].value).startswith("YES"):
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in ("00000000", "FFFFFFFF"):
                    cell.fill = purge_fill
            row[cc_col-1].font = Font(bold=True, color="7B1C1C")


def style_terminal_sheet(ws):
    """
    VT_Terminal_Status sheet styling:
      • TERMINAL rows  — darker amber header tint per row
      • PART rows      — lighter tint
      • Terminal_Status column colour-coded:
          DOWN — BLOCKING  → bright red background + bold white font
          OPERATIONAL      → green
          BLOCKED          → red
          CLEAR            → green
          NO DATA          → grey
      • Section header rows (where Section = TERMINAL or PART) get
        a bold label so the two blocks are visually separated.
    """
    header_hex  = HEADER_COLORS["VT_Terminal_Status"]
    style_sheet(ws, header_hex)

    headers      = [c.value for c in ws[1]]
    section_col  = headers.index("Section")         + 1 if "Section"         in headers else None
    status_col   = headers.index("Terminal_Status") + 1 if "Terminal_Status" in headers else None

    term_row_fill = PatternFill("solid", fgColor="FFF2CC")   # light amber — terminal rows
    part_row_fill = PatternFill("solid", fgColor="EFF6FF")   # light blue  — part rows
    down_fill     = PatternFill("solid", fgColor="FF0000")   # bright red
    oper_fill     = PatternFill("solid", fgColor="C6EFCE")   # green
    blkd_fill     = PatternFill("solid", fgColor="FFC7CE")   # pink-red
    clear_fill    = PatternFill("solid", fgColor="C6EFCE")   # green
    nodata_fill   = PatternFill("solid", fgColor="EDEDED")   # grey

    status_fill_map = {
        "DOWN — BLOCKING": down_fill,
        "OPERATIONAL":     oper_fill,
        "BLOCKED":         blkd_fill,
        "CLEAR":           clear_fill,
        "NO DATA":         nodata_fill,
    }

    for row in ws.iter_rows(min_row=2):
        section = str(row[section_col - 1].value).strip() if section_col else ""
        # Row background
        row_fill = term_row_fill if section == "TERMINAL" else part_row_fill
        for cell in row:
            val = str(cell.value) if cell.value is not None else ""
            if val == "":
                continue   # blank separator row — leave white
            if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in ("00000000", "FFFFFFFF"):
                cell.fill = row_fill

        # Status cell override
        if status_col:
            s_cell = row[status_col - 1]
            s_val  = str(s_cell.value).strip() if s_cell.value else ""
            fill   = status_fill_map.get(s_val)
            if fill:
                s_cell.fill = fill
            if s_val == "DOWN — BLOCKING":
                s_cell.font = Font(bold=True, color="FFFFFF")
            elif s_val == "BLOCKED":
                s_cell.font = Font(bold=True, color="7B1C1C")
            elif s_val == "CLEAR":
                s_cell.font = Font(bold=True, color="375623")

    ws.freeze_panes = "A2"


def style_manual_intervention_sheet(ws):
    """
    VT_Manual_Intervention: colour entire rows by priority tier.
      P0 ★ Runner STOP LINE   → bright red row + bold
      P1 ★ Runner CRITICAL    → dark orange row + bold
      P1   non-Runner          → orange-red row + bold
      P2 ★ Runner WARNING      → amber row
      P2/P3 non-Runner        → yellow row
    Also colour Terminal_Criticality and Terminal_Coverage_% columns.
    """
    style_sheet(ws, HEADER_COLORS["VT_Manual_Intervention"])

    headers      = [c.value for c in ws[1]]
    priority_col = headers.index("Priority")             + 1 if "Priority"             in headers else None
    crit_col     = headers.index("Terminal_Criticality") + 1 if "Terminal_Criticality" in headers else None
    cov_col      = headers.index("Terminal_Coverage_%")  + 1 if "Terminal_Coverage_%"  in headers else None

    fill_p0  = PatternFill("solid", fgColor="FF0000")   # bright red
    fill_p1r = PatternFill("solid", fgColor="FF4500")   # orange-red
    fill_p1  = PatternFill("solid", fgColor="FFA500")   # orange
    fill_p2r = PatternFill("solid", fgColor="FFD700")   # gold
    fill_p2  = PatternFill("solid", fgColor="FFEB9C")   # light yellow
    fill_p3  = PatternFill("solid", fgColor="FFF2CC")   # pale yellow

    for row in ws.iter_rows(min_row=2):
        if not priority_col:
            continue
        priority = str(row[priority_col - 1].value or "").strip()

        if "P0" in priority:
            fill, bold, font_color = fill_p0, True, "FFFFFF"
        elif "P1 ★" in priority:
            fill, bold, font_color = fill_p1r, True, "FFFFFF"
        elif "P1 —" in priority:
            fill, bold, font_color = fill_p1, True, "000000"
        elif "P2 ★" in priority:
            fill, bold, font_color = fill_p2r, True, "000000"
        elif "P2 —" in priority:
            fill, bold, font_color = fill_p2, False, "000000"
        else:
            fill, bold, font_color = fill_p3, False, "000000"

        for cell in row:
            cell.fill = fill
            cell.font = Font(bold=bold, color=font_color)

        # Override Terminal_Criticality cell colour
        if crit_col:
            c = row[crit_col - 1]
            cval = str(c.value or "")
            if "HIGHLY" in cval:
                c.fill = PatternFill("solid", fgColor="FF0000")
                c.font = Font(bold=True, color="FFFFFF")
            elif cval == "CRITICAL":
                c.fill = PatternFill("solid", fgColor="FFC7CE")
                c.font = Font(bold=True, color="7B1C1C")
            elif cval == "BLOCKED":
                c.fill = PatternFill("solid", fgColor="000000")
                c.font = Font(bold=True, color="FFFFFF")

        # Colour Terminal_Coverage_% cell
        if cov_col:
            c = row[cov_col - 1]
            try:
                pct = float(c.value)
                if pct < TERMINAL_HIGHLY_CRITICAL_PCT:
                    c.fill = PatternFill("solid", fgColor="FF0000")
                    c.font = Font(bold=True, color="FFFFFF")
                elif pct < TERMINAL_CRITICAL_PCT:
                    c.fill = PatternFill("solid", fgColor="FFC7CE")
            except (TypeError, ValueError):
                pass

    ws.freeze_panes = "B2"
print(f"\nWriting output → {output_path}")

vt_mw   = build_machine_wise_plan(vt_plan)
vt_co_q = build_co_queue(
    [{k:v for k,v in r.items()} for r in vt_plan.to_dict("records")]
    if not vt_plan.empty else [],
    vt_machines
)

# ── Terminal status report sheet ───────────────────────────────
def build_terminal_status_sheet():
    """
    Section A — Terminal-level view:
      One row per terminal with opening stock, planned consumption,
      closing stock, criticality level, and which parts it constrains.

    Section B — Part-level view:
      One row per part showing terminal status, criticality,
      max producible qty vs daily indent coverage %.
    """
    rows = []

    # ── Section A: terminal rows ─────────────────────────────────
    all_term_ids = set()
    for terms in part_terminals.values():
        all_term_ids.update(terms)

    for tid in sorted(all_term_ids):
        opening  = terminal_stock.get(tid, 0.0)
        closing  = terminal_running_stock.get(tid, opening)
        consumed = round(opening - closing, 0)
        is_down  = (opening <= 0)

        # Which parts use this terminal and their criticality
        parts_using = sorted(p for p, ts in part_terminals.items() if tid in ts)
        parts_blocked = [p for p in parts_using if opening <= 0]

        # Worst coverage % across all parts using this terminal
        worst_pct = 100.0
        for p in parts_using:
            d = indent_daily.get(p, 0)
            if d > 0:
                worst_pct = min(worst_pct, (opening / d) * 100)

        if is_down:
            t_status = "DOWN — BLOCKING"
        elif worst_pct < TERMINAL_HIGHLY_CRITICAL_PCT:
            t_status = "HIGHLY CRITICAL"
        elif worst_pct < TERMINAL_CRITICAL_PCT:
            t_status = "CRITICAL"
        else:
            t_status = "OPERATIONAL"

        rows.append({
            "Section":              "TERMINAL",
            "Terminal_ID":          tid,
            "Part":                 "—",
            "Opening_Stock":        round(opening, 0),
            "Planned_Consumption":  consumed,
            "Closing_Stock":        round(closing, 0),
            "Worst_Coverage_%":     round(worst_pct, 1) if worst_pct < 1e9 else 999,
            "Terminal_Status":      t_status,
            "Parts_Using":          ", ".join(parts_using),
            "Parts_Blocked":        ", ".join(parts_blocked) or "—",
            "Reason":               terminal_detail.get(tid, {}).get("reason", "") or "—",
        })

    if rows:
        rows.append({k: "" for k in rows[0].keys()})   # blank separator

    # ── Section B: part rows ─────────────────────────────────────
    for p in sorted(part_terminals.keys()):
        req    = part_terminals.get(p, [])
        daily  = indent_daily.get(p, 0.0)
        inv    = inventory.get(p, 0.0)
        cat    = part_category.get(p, "Stranger")

        t_level, lim_term, max_prod = terminal_criticality(p, use_running_stock=False)
        cov_pct = round((max_prod / daily) * 100, 1) if daily > 0 and max_prod != float("inf") else 100.0

        # Terminal stocks for all required terminals
        term_stocks = {t: terminal_stock.get(t, "N/A") for t in req}
        term_closing = {t: terminal_running_stock.get(t, terminal_stock.get(t, 0)) for t in req}

        is_runner_critical = (
            cat == "Runner"
            and t_level in ("BLOCKED", "HIGHLY CRITICAL", "CRITICAL")
            and daily > 0
            and inv < SAFETY_DAYS * daily
        )

        rows.append({
            "Section":              "PART",
            "Terminal_ID":          "—",
            "Part":                 p,
            "Opening_Stock":        "—",
            "Planned_Consumption":  "—",
            "Closing_Stock":        "—",
            "Worst_Coverage_%":     round(cov_pct, 1),
            "Terminal_Status":      t_level,
            "Parts_Using":          "—",
            "Parts_Blocked":        "—",
            "Reason":               (
                f"Limiting: {lim_term}  "
                f"| Max producible: {max_prod:.0f}  "
                f"| Daily indent: {daily:.0f}  "
                f"| Coverage: {cov_pct:.1f}%"
                + ("  ⚠ RUNNER + LOW INV — MANUAL INTERVENTION NEEDED" if is_runner_critical else "")
            ),
        })

    return pd.DataFrame(rows) if rows else pd.DataFrame()


def build_manual_intervention_sheet(plan_df):
    """
    Flags ALL parts (any category) that are:
      (a) Terminal-BLOCKED with inventory < SAFETY_DAYS * daily  → URGENT
      (b) Terminal CRITICAL / HIGHLY CRITICAL with inv < SAFETY_DAYS * daily → WARNING

    Runner parts in any of the above situations are marked HIGHEST PRIORITY.

    Each row gives:
      • Part, Category, Inventory days coverage
      • Terminal criticality level + limiting terminal
      • Opening stock of limiting terminal
      • Max producible qty and gap to daily indent
      • Recommended action
    """
    rows = []
    all_parts_list = list(data["Material"].unique())

    for part in sorted(all_parts_list):
        daily   = indent_daily.get(part, 0.0)
        inv     = inventory.get(part, 0.0)
        cat     = part_category.get(part, "Stranger")
        monthly = indent_monthly.get(part, 0.0)
        r_val   = rate.get(part, 0.0)

        if daily <= 0 or monthly <= 0 or r_val <= 0:
            continue

        t_level, lim_term, max_prod = terminal_criticality(part, use_running_stock=False)
        inv_days = inv / daily if daily > 0 else 0.0

        is_blocked   = (t_level == "BLOCKED")
        is_critical  = (t_level in ("HIGHLY CRITICAL", "CRITICAL"))
        low_inv      = (inv < SAFETY_DAYS * daily)
        zero_inv     = (inv == 0)

        # Only flag if there's a real terminal problem AND stock is concerning
        if not (is_blocked or is_critical):
            continue
        if not low_inv and not zero_inv:
            continue   # plenty of stock — terminal shortage not urgent today

        lim_stock = terminal_stock.get(lim_term, 0.0) if lim_term != "—" else 0.0
        prod_gap  = max(0.0, daily - max_prod) if max_prod != float("inf") else 0.0
        cov_pct   = round((max_prod / daily) * 100, 1) if daily > 0 and max_prod != float("inf") else 100.0

        if is_blocked and zero_inv:
            priority   = "P1 — STOP LINE RISK"
            action     = (f"IMMEDIATE: Part cannot run (terminal stock=0) AND inventory=0. "
                          f"Source {lim_term} urgently or escalate.")
        elif is_blocked and low_inv:
            priority   = "P1 — CRITICAL"
            action     = (f"URGENT: Terminal {lim_term} stock=0. "
                          f"Inv covers {inv_days:.1f} days only. "
                          f"Source terminals immediately.")
        elif t_level == "HIGHLY CRITICAL" and zero_inv:
            priority   = "P1 — HIGHLY CRITICAL + ZERO INV"
            action     = (f"URGENT: Only {lim_stock:.0f} terminals available "
                          f"(covers {cov_pct:.1f}% of daily indent). "
                          f"Inventory is zero. Prioritise terminal procurement.")
        elif t_level == "HIGHLY CRITICAL":
            priority   = "P2 — HIGHLY CRITICAL"
            action     = (f"Terminal {lim_term} covers only {cov_pct:.1f}% of daily indent "
                          f"({lim_stock:.0f} available vs {daily:.0f} needed). "
                          f"Plan partial run of {max_prod:.0f} pcs. Inv covers {inv_days:.1f} days.")
        else:   # CRITICAL
            priority   = "P3 — CRITICAL"
            action     = (f"Terminal {lim_term} covers {cov_pct:.1f}% of daily indent "
                          f"({lim_stock:.0f} available vs {daily:.0f} needed). "
                          f"Partial run: {max_prod:.0f} pcs. Shortfall: {prod_gap:.0f} pcs. "
                          f"Monitor closely.")

        if cat == "Runner":
            priority = priority.replace("P1", "P0 ★").replace("P2", "P1 ★").replace("P3", "P2 ★")

        rows.append({
            "Priority":             priority,
            "Part":                 part,
            "Category":             cat,
            "Color":                part_color.get(part, "UNKNOWN"),
            "Inv_Scenario":         _inv_scenario_label(part, inventory),
            "Inventory_Now":        round(inv, 0),
            "Daily_Indent":         round(daily, 0),
            "Days_Coverage":        round(inv_days, 2),
            "Safety_Floor_Days":    SAFETY_DAYS,
            "Terminal_Criticality": t_level,
            "Limiting_Terminal":    lim_term,
            "Terminal_Opening_Stock": round(lim_stock, 0) if lim_term != "—" else "—",
            "Max_Producible_Today": round(max_prod, 0) if max_prod != float("inf") else "Unlimited",
            "Terminal_Coverage_%":  cov_pct,
            "Shortfall_vs_Daily":   round(prod_gap, 0),
            "Required_Terminals":   terminals_for_display(part),
            "Recommended_Action":   action,
        })

    # Sort: P0 first, then P1, P2, P3, then by days coverage ascending
    priority_order = {"P0 ★": 0, "P1 ★": 1, "P2 ★": 2,
                      "P1 — STOP LINE RISK": 3, "P1 — CRITICAL": 4,
                      "P1 — HIGHLY CRITICAL + ZERO INV": 5,
                      "P2 — HIGHLY CRITICAL": 6, "P3 — CRITICAL": 7}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort_p"] = df["Priority"].apply(lambda x: next(
            (v for k, v in priority_order.items() if k in x), 99))
        df = df.sort_values(["_sort_p", "Days_Coverage"]).drop(columns=["_sort_p"]).reset_index(drop=True)
    return df


vt_terminal_status      = build_terminal_status_sheet()
vt_manual_intervention  = build_manual_intervention_sheet(vt_plan)

sheets = {
    "VT_Plan_By_Machine":      vt_mw,
    "VT_CO_Queue":             vt_co_q,
    "VT_Plan":                 vt_plan,
    "VT_Manual_Intervention":  vt_manual_intervention,  # ← NEW: terminal alert sheet
    "VT_Terminal_Status":      vt_terminal_status,
    "VT_Inventory_Target":     vt_inv_target,
    "VT_Multi_Machine_Parts":  vt_multi_machine,
    "VT_Production_vs_Indent": vt_prod_vs_indent,
    "VT_Daily_Indent_Status":  vt_indent_status,
    "VT_Priority_Scores":      vt_scores,
    "VT_Machine_Util":         vt_mach,
    "VT_Not_Planned":          vt_not,
    "VT_Deferred":             vt_def,
    "VT_Inventory_Health":     vt_inv,
    "VT_Indent_Horizon":       vt_horizon,
    "VT_Part_Audit":           audit_df,
}
if not vt_micro.empty:
    sheets["VT_Micro_Idle"] = vt_micro

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine"      in wb.sheetnames: style_machine_wise_sheet(wb["VT_Plan_By_Machine"])
if "VT_Production_vs_Indent" in wb.sheetnames: style_prod_vs_indent_sheet(wb["VT_Production_vs_Indent"])
if "VT_Multi_Machine_Parts"  in wb.sheetnames: style_multi_machine_sheet(wb["VT_Multi_Machine_Parts"])
if "VT_Inventory_Target"     in wb.sheetnames: style_inv_target_sheet(wb["VT_Inventory_Target"])
if "VT_CO_Queue"             in wb.sheetnames: style_co_queue_sheet(wb["VT_CO_Queue"])
if "VT_Terminal_Status"      in wb.sheetnames: style_terminal_sheet(wb["VT_Terminal_Status"])
if "VT_Manual_Intervention"  in wb.sheetnames: style_manual_intervention_sheet(wb["VT_Manual_Intervention"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name not in (
            "VT_Plan_By_Machine", "VT_Production_vs_Indent",
            "VT_Multi_Machine_Parts", "VT_Inventory_Target",
            "VT_CO_Queue", "VT_Terminal_Status",
            "VT_Manual_Intervention"):
        style_sheet(wb[sheet_name], header_hex)

if "VT_Daily_Indent_Status" in wb.sheetnames:
    ws_is      = wb["VT_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    meets_col  = headers_is.index("Meets_Daily_Indent") + 1 if "Meets_Daily_Indent" in headers_is else None
    covers_col = headers_is.index("Covers_With_Inv")    + 1 if "Covers_With_Inv"    in headers_is else None
    for row in ws_is.iter_rows(min_row=2):
        if meets_col:
            cell = row[meets_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE") if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFC7CE"))
        if covers_col:
            cell = row[covers_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE") if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFEB9C"))
        if meets_col and str(row[meets_col - 1].value) == "NO ✗":
            for cell in row:
                cell.font = Font(bold=True)

for name, color in HEADER_COLORS.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")


# =============================================================
# FINAL SUMMARY
# =============================================================

print(f"\n{'='*65}")
print(f"  Smart APS V9 Complete  —  {PLANNING_DATE}")
print(f"  Safety floor  : {SAFETY_DAYS} days  |  Target ceiling: {TARGET_DAYS} days")
print(f"  Util soft cap : {UTIL_TARGET_PCT}%  (breakdown buffer: {100-UTIL_TARGET_PCT:.0f}% = {AVAILABLE_HOURS*(100-UTIL_TARGET_PCT)/100:.1f}h)")
print(f"  Colour purge  : ACTIVE  |  Purge penalty: {int(COLOR_PURGE_HRS*60)} min")
print(f"  CHANGES ACTIVE:")
print(f"    [1] Skip: daily ≤{MIN_DAILY_INDENT} OR monthly ≤{MIN_INDENT_HOURS}h")
print(f"    [2] Repeater/Stranger → single tool only")
print(f"    [3] Repeater/Stranger → single machine only (no splits)")
print(f"    [4] Utilisation soft ceiling: {UTIL_TARGET_PCT}% (breakdown buffer)")
print(f"    [5] Runner deferral: skip if inv ≥ {RUNNER_DEFER_INV_DAYS} days (plan tomorrow)")
print(f"    [6] S0/S1/S2/S3 per-part OPD cap framework")
print(f"    [7] Terminal gate: parts blocked if any required terminal is down")
print(f"{'='*65}")

# Terminal summary
_terminal_blocked_count = sum(1 for s in audit_df["Status"] if s == "TERMINAL BLOCKED")
_manual_count           = len(vt_manual_intervention) if not vt_manual_intervention.empty else 0

print(f"\n  Terminal gate summary  (1 terminal consumed per piece):")
print(f"    Terminals with zero stock       : {len(unavailable_terminals)}")
print(f"    Parts blocked by terminal       : {_terminal_blocked_count}")
print(f"    Manual intervention alerts      : {_manual_count}")

if terminal_stock:
    print(f"\n  Terminal stock movement (opening → closing):")
    for tid in sorted(terminal_stock.keys()):
        opening  = terminal_stock.get(tid, 0.0)
        closing  = terminal_running_stock.get(tid, opening)
        consumed = round(opening - closing, 0)
        flag = ""
        # Show criticality flag
        parts_using = [p for p, ts in part_terminals.items() if tid in ts]
        worst_pct = 100.0
        for p in parts_using:
            d = indent_daily.get(p, 0)
            if d > 0:
                worst_pct = min(worst_pct, (opening / d) * 100)
        if opening <= 0:
            flag = "  ✗ ZERO STOCK — BLOCKING"
        elif worst_pct < TERMINAL_HIGHLY_CRITICAL_PCT:
            flag = f"  ⚡ HIGHLY CRITICAL ({worst_pct:.1f}% coverage)"
        elif worst_pct < TERMINAL_CRITICAL_PCT:
            flag = f"  ⚠ CRITICAL ({worst_pct:.1f}% coverage)"
        print(f"    {tid:<22}  open={opening:>8.0f}  consumed={consumed:>8.0f}  "
              f"close={closing:>8.0f}{flag}")

if _manual_count > 0:
    print(f"\n  ⚠ MANUAL INTERVENTION REQUIRED — {_manual_count} part(s) need attention:")
    for _, row in vt_manual_intervention.iterrows():
        print(f"    [{row['Priority']:<30}] {row['Part']:<30} "
              f"terminal={row['Limiting_Terminal']:<20} "
              f"coverage={row['Terminal_Coverage_%']}%  "
              f"inv={row['Days_Coverage']}d")

for status, count in audit_df["Status"].value_counts().items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<60}: {count:>4}")

print(f"\n  Results:")
print(f"    Planned            : {len(vt_plan):>4} rows")
print(f"    Not planned        : {len(vt_not):>4}")
print(f"    Deferred           : {len(vt_def):>4}")

if not vt_inv_target.empty:
    at_t = (vt_inv_target["Buffer_Status"] == "AT_TARGET").sum()
    bld  = (vt_inv_target["Buffer_Status"] == "BUILDING").sum()
    bls  = (vt_inv_target["Buffer_Status"] == "BELOW_SAFETY").sum()
    crt  = (vt_inv_target["Buffer_Status"] == "CRITICAL").sum()
    print(f"\n  Inventory target status (end of day):")
    print(f"    S3-HEALTHY  AT_TARGET   (≥5 days) : {at_t:>4}  — skip tomorrow")
    print(f"    S2-BELOW    BUILDING  (3–5 days)  : {bld:>4}  — gradual build continues")
    print(f"    S1-MIXED    BELOW_SAFETY (<3 days): {bls:>4}  — safety buffer being consumed")
    print(f"    S0-CRITICAL CRITICAL  (0 pcs)     : {crt:>4}  — displacement eligible tomorrow")

if not vt_mach.empty:
    print(f"\n  Machine utilization (90% soft ceiling):")
    print(f"    Average util    : {vt_mach['Utilization_%'].mean():.1f}%")
    print(f"    UNDERUSED       : {(vt_mach['Status']=='UNDERUSED').sum()} machines")
    print(f"    Avg brkdwn buf  : {vt_mach['Breakdown_Buffer_Hrs'].mean():.2f}h per machine")
    print(f"    Total purges    : {vt_mach['Color_Purges'].sum()} colour changeovers today")

print(f"\n  Colour groups today:")
for col, pts in sorted(color_groups.items()):
    print(f"    {col:<20} : {len(pts):>3} parts")

print(f"\n  Output → {output_path}")
print(f"  State  → {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date(2026, 3, 21)")
print(f"{'='*65}")


  Smart APS V9  —  5-Day Target Inventory System
  Planning date : 2026-03-20
  Indent month  : March 2026
  Working days  : 26  (31 days − 5 Sundays)
  Safety floor  : 3 days  |  Target ceiling : 5 days
  Util soft cap : 90.0%  (breakdown buffer)
  Terminal gate : ACTIVE — parts blocked if any required terminal is unavailable

Loading data...
  Terminal data loaded  ✓  (158 part rows, 196 unavailable terminal records)
  VT parts in sheet       : 158
  Parts with valid rate   : 153
  Distinct colours        : 5
    BLACK                → 107 part(s)
    BLUE                 → 8 part(s)
    BROWN                → 4 part(s)
    GREEN                → 2 part(s)
    WHITE                → 37 part(s)

  Terminal system summary  (consumption model: 1 terminal per piece):
    Parts mapped to terminals     : 0
    Parts with no terminal listed : 158  (unrestricted)
    Terminals with zero stock     : 40
    Criticality thresholds        : HIGHLY CRITICAL <75.0%  |  CRITICAL <80.0%  of daily i